# 🗑️ Automatic Litter Detection & Instance Segmentation using EfficientSAM — FINAL
**ทีม อุดมไปด้วยธาตุเหล็ก · CP461 Computer Vision · Final project**

**Pipeline:** TACO photos → **YOLO11 detector (trained here, 5 classes)** → bounding boxes as prompts → **EfficientSAM-Ti / -S (pretrained, zero-shot)** → instance masks + class + count

| | |
|---|---|
| Paper | *EfficientSAM: Leveraged Masked Image Pretraining for Efficient Segment Anything*, Xiong et al., CVPR 2024 |
| Code | [yformer/EfficientSAM](https://github.com/yformer/EfficientSAM) · [ultralytics/ultralytics](https://github.com/ultralytics/ultralytics) |
| Data | [TACO](https://github.com/pedropro/TACO): 1,500 images, 4,784 objects, 60 categories regrouped into 5 classes |

### ▶️ How to run (Colab)
1. **Runtime ▸ Change runtime type ▸ T4 GPU**.
2. Keep `DRIVE_FOLDER` = the **same Drive folder as the progress run** (default `CP461Project`). Images, the v1 detector and the YOLO-seg baseline are reused from it, nothing is downloaded or retrained twice.
3. **Runtime ▸ Run all** and keep the tab open. Time on a T4 (with the progress run in Drive): **≈ 1.5–2 h**: detector v2 training ≈ 60–75 min, evaluation + E6/E7 ≈ 20–25 min. From an empty Drive folder add ≈ 1.5 h (download + v1 + YOLO-seg).
4. If Colab disconnects: **Run all** again — training resumes from the last checkpoint.
5. All figures and tables are saved in `outputs/` in your Drive folder and downloaded as **`outputs.zip`** at the end.

### What changed since the progress report (answers to the instructor's comments)
| Comment | Answer in this notebook |
|---|---|
| "Confusion matrix still not good" | **E1b** detector v2: 960 px + repeat-factor sampling for rare classes; v1 vs v2 confusion matrices (5.2) |
| "Class imbalance → accuracy alone is not enough" | **E7** macro-F1, balanced accuracy, Cohen's κ, MCC, per-class P/R/F1, object-level ROC-AUC / PR-AUC, per-class thresholds |
| "What does *studying the effect of the prompt* mean? Report it" | **E6** prompt study: box vs point vs box+points vs noisy boxes, by class, size and object shape |
| "What is the benchmark? Higher or lower?" | **12.5** benchmark table vs TACO paper, other TACO detectors and the EfficientSAM paper |

### Experiments
| # | Experiment | Question it answers | Metrics |
|---|---|---|---|
| E1 / E1b | YOLO detector v1 (progress) vs v2 (final) | How well do we find and classify litter? Did v2 fix the confusion matrix? | P, R, F1, mAP50, mAP50-95, confusion matrix |
| E2 | **GT boxes** → EfficientSAM-Ti vs -S | How good are masks with a perfect box? | IoU, Dice (per class, size) |
| E3 | **YOLO boxes** → EfficientSAM-Ti vs -S | Full pipeline quality | Mask AP, AP50, AP75, IoU, Dice |
| E4 | YOLO-seg baseline (one-stage) | Is Detector + EfficientSAM better than an end-to-end segmenter? | Same as E3 |
| E5 | Speed | What does each option cost? | ms / image, parameters |
| E6 | **Prompt study** | Which prompt gives the best masks, and why do thin objects fail? | mIoU, IoU≥0.5/0.75 rate by class, size, fill ratio, box noise |
| E7 | **Imbalance-aware evaluation** | Is the detector good on *every* class, not only Plastic? | macro/weighted F1, balanced acc., κ, MCC, ROC-AUC, PR-AUC |

## 0 · Configuration
The defaults work as they are. Change these only if you want to experiment.

In [ ]:
# =============================== CONFIG ===============================
PROJECT_NAME       = "litter_esam"
SAVE_TO_DRIVE      = True             # keep images, weights and results in Google Drive (survives disconnects)
DRIVE_FOLDER       = "CP461Project"    # your folder inside "My Drive", e.g. "CP461/Litter Project" (created if missing)
RETRAIN            = False            # True = delete the saved weights and train again once (then set back to False)

DET_MODEL          = "yolo11s.pt"     # detector to fine-tune: yolo11n / s / m / l / x .pt
SEG_MODEL          = "yolo11s-seg.pt" # one-stage baseline for comparison (E4)
TRAIN_SEG_BASELINE = True             # False skips E4 and saves ~40 min
EPOCHS             = 80               # detector epochs
SEG_EPOCHS         = 80               # YOLO-seg baseline epochs
PATIENCE           = 20               # early stopping (epochs without val improvement)
IMG_SIZE           = 640              # training size; 960 helps tiny litter (cigarettes, caps) but is ~2x slower
BATCH              = 16
MAX_SIDE           = 1280             # TACO photos are up to 6000 px: resize the long side to this once
SPLIT              = (0.70, 0.15, 0.15)   # train / val / test, split by image
SEED               = 42

SAM_VARIANTS       = ["ti", "s"]      # EfficientSAM-Ti and EfficientSAM-S
CONF_THRES         = 0.25             # confidence used for counting, confusion matrix, matched IoU, figures
MATCH_IOU          = 0.5              # a detection is "correct" if its box IoU with a GT box is >= this
MAX_TEST_IMAGES    = None             # None = whole test split (e.g. 40 for a quick dry run)
TIMING_IMAGES      = 50               # images used for the speed benchmark (E5)
LAUNCH_DEMO        = True             # Gradio web demo (prototype) at the end

# ---- final: detector v2 (answers "the confusion matrix is still not good") ----
TRAIN_DET_V2       = True             # False = keep the progress-report detector (v1) for everything
DET_V2_MODEL       = "yolo11s.pt"     # same size as v1, so the gain comes from the recipe, not a bigger model
DET_V2_IMG_SIZE    = 960              # larger input for small litter (v1 used 640)
DET_V2_RFS         = True             # repeat-factor sampling: show images with rare classes more often
RFS_T              = 0.3              # RFS threshold t (classes in < 30% of training images get repeated)
DET_V2_EPOCHS      = 80
DET_V2_BATCH       = 16               # T4 16 GB is fine at 960 px; use 8 if you get CUDA out-of-memory

## 1 · Setup
Installs Ultralytics + pycocotools and clones the EfficientSAM and TACO repositories.

In [ ]:
%pip install -q "ultralytics==8.4.175" pycocotools tqdm

In [ ]:
import os, sys, io, json, time, math, random, shutil, zipfile, subprocess, contextlib, warnings, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, matplotlib.patches, matplotlib.colors, cv2, yaml, torch
import torch.nn.functional as F
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = DRIVE_FOLDER.strip().strip("/")
    for prefix in ("content/drive/MyDrive/", "drive/MyDrive/", "MyDrive/", "My Drive/"):
        folder = folder[len(prefix):] if folder.startswith(prefix) else folder
    WORK = Path("/content/drive/MyDrive") / folder
else:
    WORK = BASE / PROJECT_NAME
# Saved (WORK):   data/images + annotations, runs/ (weights, training curves), outputs/ (tables, figures, zip)
# Not saved:      repos (re-cloned in seconds) and LOCAL, a fast working copy of the images rebuilt each session
REPOS, LOCAL = BASE / "repos", BASE / "litter_local"
DATA, RUNS, OUT = WORK / "data", WORK / "runs", WORK / "outputs"
FIG = OUT / "figures"
for p in (REPOS, LOCAL, DATA, RUNS, FIG):
    p.mkdir(parents=True, exist_ok=True)
print("💾 Everything is saved in:", WORK)

def clone(url, dst):
    if not dst.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(dst)], check=True)
    print("✓", dst)

clone("https://github.com/yformer/EfficientSAM.git", REPOS / "EfficientSAM")
clone("https://github.com/pedropro/TACO.git", REPOS / "TACO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_DEVICE = 0 if DEVICE == "cuda" else "cpu"

def free_gpu():
    # release GPU memory still held by finished steps (e.g. the YOLO trainers) before the next big step
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu":
    print("⚠️  No GPU found → Runtime ▸ Change runtime type ▸ T4 GPU, then Runtime ▸ Run all.")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- class list + plotting style (fixed-order, colour-blind-checked categorical palette) ---
CLASSES = ["Plastic", "Paper & Cardboard", "Metal", "Glass", "Other Waste"]
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
METHOD_COLORS = {"EfficientSAM-Ti": "#2a78d6", "EfficientSAM-S": "#eb6834", "YOLO-seg": "#1baf7a"}
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "legend.frameon": False,
})

def savefig(name):
    plt.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.show()
    print("saved →", FIG / name)

def hex2rgb(h):
    return tuple(int(h.lstrip("#")[k:k + 2], 16) for k in (0, 2, 4))

## 2 · Data: TACO → 5 classes
TACO's 60 categories are regrouped by material into **Plastic, Paper & Cardboard, Metal, Glass, Other Waste**.

In [ ]:
P, PA, M, G, O = CLASSES
TACO_TO_5 = {
    # Plastic
    "Other plastic bottle": P, "Clear plastic bottle": P, "Plastic bottle cap": P, "Disposable plastic cup": P,
    "Foam cup": P, "Other plastic cup": P, "Plastic lid": P, "Other plastic": P, "Plastic film": P,
    "Six pack rings": P, "Garbage bag": P, "Other plastic wrapper": P, "Single-use carrier bag": P,
    "Polypropylene bag": P, "Crisp packet": P, "Spread tub": P, "Tupperware": P, "Disposable food container": P,
    "Foam food container": P, "Other plastic container": P, "Plastic glooves": P, "Plastic utensils": P,
    "Squeezable tube": P, "Plastic straw": P, "Styrofoam piece": P,
    # Paper & Cardboard
    "Toilet tube": PA, "Other carton": PA, "Egg carton": PA, "Drink carton": PA, "Corrugated carton": PA,
    "Meal carton": PA, "Pizza box": PA, "Paper cup": PA, "Magazine paper": PA, "Tissues": PA,
    "Wrapping paper": PA, "Normal paper": PA, "Paper bag": PA, "Plastified paper bag": PA, "Paper straw": PA,
    # Metal
    "Aluminium foil": M, "Aluminium blister pack": M, "Metal bottle cap": M, "Food Can": M, "Aerosol": M,
    "Drink can": M, "Metal lid": M, "Pop tab": M, "Scrap metal": M,
    # Glass
    "Glass bottle": G, "Broken glass": G, "Glass cup": G, "Glass jar": G,
    # Other Waste
    "Battery": O, "Carded blister pack": O, "Food waste": O, "Rope & strings": O, "Shoe": O,
    "Unlabeled litter": O, "Cigarette": O,
}
taco = json.load(open(REPOS / "TACO/data/annotations.json"))
unmapped = [c["name"] for c in taco["categories"] if c["name"] not in TACO_TO_5]
assert not unmapped, f"unmapped TACO categories: {unmapped}"
cat2cls = {c["id"]: CLASSES.index(TACO_TO_5[c["name"]]) for c in taco["categories"]}
cat_name = {c["id"]: c["name"] for c in taco["categories"]}
print(f"TACO: {len(taco['images'])} images · {len(taco['annotations'])} objects · {len(taco['categories'])} categories → {len(CLASSES)} classes")
mapping = (pd.DataFrame([(TACO_TO_5[c["name"]], c["name"]) for c in taco["categories"]], columns=["class", "TACO categories"])
             .groupby("class", sort=False)["TACO categories"].apply(", ".join).reindex(CLASSES).to_frame())
mapping.to_csv(OUT / "class_mapping.csv"); display(mapping)

### 2.1 Download the images (≈10–15 min)
Downloads every photo from Flickr in parallel, fixes EXIF rotation (TACO annotations follow the EXIF orientation), resizes the long side to `MAX_SIDE`, and scales the annotations to match. Re-running the cell only fetches what is missing.

In [ ]:
import requests
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
Image.MAX_IMAGE_PIXELS = None
IMG_DIR, META_DIR = DATA / "images", DATA / f"meta_{MAX_SIDE}"
IMG_DIR.mkdir(exist_ok=True); META_DIR.mkdir(exist_ok=True)

GONE = {403, 404, 410}                                    # Flickr says the photo is private or deleted: don't retry

def get_image(url):
    # Download + decode one image with a few polite retries -> (PIL image or None, error text, permanently gone?)
    err = ""
    for attempt in range(3):
        wait = 2 * (attempt + 1)
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                im = Image.open(io.BytesIO(r.content)); im.load()
                return im, "", False
            err = f"HTTP {r.status_code}"
            if r.status_code in GONE:
                return None, err, True
            try:
                wait = max(wait, float(r.headers.get("Retry-After", 0)))
            except ValueError:
                pass
        except Exception as e:
            err = repr(e)[:120]
        time.sleep(min(wait, 20))
    return None, err, False

def fetch(img):
    meta_f = META_DIR / f"{img['id']}.json"
    if meta_f.exists():
        meta = json.loads(meta_f.read_text())
        if (IMG_DIR / meta["file"]).exists():
            return meta                                    # already downloaded in an earlier run (kept in Drive)
    W0, H0 = img["width"], img["height"]
    local = REPOS / "TACO/data" / img["file_name"]        # reuse photos from TACO's own download.py, if present
    try:
        if local.exists():
            im, src = Image.open(local), "local"; im.load()
        else:
            im, src, errs, gone = None, "", [], True
            # original photo first; if Flickr refuses it, its 640 px copy (annotations are rescaled to match)
            for url, tag in [(img.get("flickr_url"), "original"), (img.get("flickr_640_url"), "640px")]:
                if not url:
                    continue
                im, err, is_gone = get_image(url)
                if im is not None:
                    src = tag; break
                errs.append(f"{tag}: {err}"); gone = gone and is_gone
            if im is None:
                return {"id": img["id"], "ok": False, "err": " · ".join(errs), "gone": gone}
        rot = ImageOps.exif_transpose(im)
        if rot.size == (W0, H0):
            im = rot                                       # usual case: annotations follow EXIF orientation
        elif im.size == (W0, H0):
            pass                                           # annotations were made on the raw pixels
        elif abs(rot.size[0] / rot.size[1] - W0 / H0) < 0.01:
            im = rot                                       # same shape, different resolution
        else:
            return {"id": img["id"], "ok": False, "err": f"size {rot.size} != annotation {(W0, H0)}"}
        im = im.convert("RGB")
        s = min(1.0, MAX_SIDE / max(im.size))
        if s < 1:
            im = im.resize((round(im.size[0] * s), round(im.size[1] * s)), Image.LANCZOS)
        name = img["file_name"].replace("/", "_").rsplit(".", 1)[0] + ".jpg"
        im.save(IMG_DIR / name, quality=95)
        meta = {"id": img["id"], "ok": True, "file": name, "w": im.size[0], "h": im.size[1],
                "sx": im.size[0] / W0, "sy": im.size[1] / H0, "src": src}
        meta_f.write_text(json.dumps(meta))
        return meta
    except Exception as e:
        return {"id": img["id"], "ok": False, "err": repr(e)}

with ThreadPoolExecutor(max_workers=8) as ex:            # 8 parallel downloads: gentler on Flickr than 16
    metas = list(tqdm(ex.map(fetch, taco["images"]), total=len(taco["images"]), desc="TACO images"))
ready = {m["id"]: m for m in metas if m["ok"]}
failed = [m for m in metas if not m["ok"]]
gone = [m for m in failed if m.get("gone")]
print(f"✓ {len(ready)} / {len(taco['images'])} images ready "
      f"({dict(Counter(m.get('src', 'original') for m in ready.values()))}) · {len(failed)} failed, {len(gone)} of them deleted/private on Flickr")
if len(failed) > len(gone):
    print("first failures:", [m["err"] for m in failed if not m.get("gone")][:3])
    print("→ Flickr refused some downloads (often rate-limiting). Re-run this cell to fetch only the missing ones;")
    print("  images already downloaded stay in your Drive folder.")
assert len(ready) > 0, "No images could be downloaded. Check the internet connection and re-run this cell."

# Fast local working copy (reading thousands of files from Drive is slow). Rebuilt each session, not saved.
FAST_IMG = LOCAL / "images"
FAST_IMG.mkdir(parents=True, exist_ok=True)
for m in tqdm(list(ready.values()), desc="local working copy"):
    dst = FAST_IMG / m["file"]
    if not dst.exists():
        try:
            os.link(IMG_DIR / m["file"], dst)
        except OSError:
            shutil.copy2(IMG_DIR / m["file"], dst)

### 2.2 Build 5-class COCO annotations + class-balanced train/val/test split

In [ ]:
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

images_all, anns_all = {}, []
for im in taco["images"]:
    m = ready.get(im["id"])
    if m:
        images_all[im["id"]] = {"id": im["id"], "file_name": m["file"], "width": m["w"], "height": m["h"]}
for a in taco["annotations"]:
    m = ready.get(a["image_id"])
    if not m:
        continue
    sx, sy, W, H = m["sx"], m["sy"], m["w"], m["h"]
    polys = [[round(v * (sx if k % 2 == 0 else sy), 2) for k, v in enumerate(p)]
             for p in a["segmentation"] if len(p) >= 6]
    x, y, w, h = a["bbox"]
    x0, y0, x1, y1 = max(0.0, x * sx), max(0.0, y * sy), min(W, (x + w) * sx), min(H, (y + h) * sy)
    if not polys or x1 - x0 < 1 or y1 - y0 < 1:
        continue
    anns_all.append({"id": len(anns_all) + 1, "image_id": a["image_id"], "category_id": cat2cls[a["category_id"]] + 1,
                     "bbox": [round(x0, 2), round(y0, 2), round(x1 - x0, 2), round(y1 - y0, 2)],
                     "segmentation": polys, "area": float(a["area"] * sx * sy), "iscrowd": 0,
                     "taco_name": cat_name[a["category_id"]]})
categories = [{"id": k + 1, "name": n, "supercategory": "litter"} for k, n in enumerate(CLASSES)]

# Class-balanced random split: try many shuffles and keep the one whose per-class share is closest to SPLIT.
ids_sorted = sorted(images_all)
A = np.zeros((len(ids_sorted), len(CLASSES)))
row = {i: r for r, i in enumerate(ids_sorted)}
for a in anns_all:
    A[row[a["image_id"]], a["category_id"] - 1] += 1
tot = np.maximum(A.sum(0), 1)
n_tr, n_va = round(len(ids_sorted) * SPLIT[0]), round(len(ids_sorted) * SPLIT[1])
best = None
for k in range(500):
    perm = np.random.RandomState(SEED + k).permutation(len(ids_sorted))
    parts = [perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]]
    score = sum(np.abs(A[p].sum(0) / tot - SPLIT[j]).sum() for j, p in enumerate(parts))
    if best is None or score < best[0]:
        best = (score, parts)
splits = {s: sorted(ids_sorted[r] for r in p) for s, p in zip(["train", "val", "test"], best[1])}

COCO_JSON = {}
for s, ids in splits.items():
    S = set(ids)
    d = {"images": [images_all[i] for i in ids], "annotations": [a for a in anns_all if a["image_id"] in S],
         "categories": categories}
    COCO_JSON[s] = DATA / f"coco5_{s}.json"
    COCO_JSON[s].write_text(json.dumps(d))

rows = []
for s, ids in splits.items():
    S, c = set(ids), np.zeros(len(CLASSES), int)
    for a in anns_all:
        if a["image_id"] in S:
            c[a["category_id"] - 1] += 1
    rows.append([s, len(ids), *c, c.sum()])
stats = pd.DataFrame(rows, columns=["split", "images", *CLASSES, "objects"]).set_index("split")
stats.loc["total"] = stats.sum()
stats.to_csv(OUT / "dataset_stats.csv")
display(stats)

### 2.3 Dataset overview + example images for each class

In [ ]:
def poly_to_mask(polys, h, w):
    return mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(polys, h, w))).astype(bool)

def load_rgb(img_id):
    return cv2.cvtColor(cv2.imread(str(FAST_IMG / images_all[img_id]["file_name"])), cv2.COLOR_BGR2RGB)

def overlay(img, mask, color, alpha=0.45, thick=2):
    out = img.copy()
    c = np.array(hex2rgb(color), float)
    out[mask] = (out[mask] * (1 - alpha) + c * alpha).astype(np.uint8)
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, cnts, -1, hex2rgb(color), thick)
    return out

# --- overview: objects per class + object size ---
tot_cls = stats.loc["total", CLASSES].astype(int)
order = tot_cls.sort_values().index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.1, 1]})
ax[0].barh(order, tot_cls[order], color=[CLASS_COLORS[CLASSES.index(c)] for c in order], height=0.6)
for y, c in enumerate(order):
    ax[0].text(tot_cls[c] + tot_cls.max() * 0.01, y, f"{tot_cls[c]:,}  ({tot_cls[c] / tot_cls.sum():.0%})",
               va="center", color=INK2, fontsize=9)
ax[0].set_xlim(0, tot_cls.max() * 1.25); ax[0].grid(axis="x"); ax[0].grid(axis="y", visible=False)
ax[0].set_title(f"Objects per class ({tot_cls.sum():,} objects, {len(images_all):,} images)", loc="left")
side = np.sqrt([a["area"] for a in anns_all])
ax[1].hist(side, bins=np.logspace(np.log10(max(side.min(), 2)), np.log10(side.max()), 40), color="#2a78d6",
           edgecolor="white", linewidth=0.6)
ax[1].set_xscale("log")
ticks = [t for t in (4, 8, 16, 32, 64, 128, 256, 512, 1024) if side.min() <= t <= side.max()]
ax[1].set_xticks(ticks, [str(t) for t in ticks]); ax[1].minorticks_off()
for t, lab in [(32, "small < 32²"), (96, "large > 96²")]:
    ax[1].axvline(t, color=MUTED, ls="--", lw=1); ax[1].text(t * 1.05, ax[1].get_ylim()[1] * 0.92, lab, color=INK2, fontsize=8)
small = (side < 32).mean()
ax[1].set_title(f"Object size: √area in px at {MAX_SIDE}px  ({small:.0%} are small)", loc="left")
ax[1].set_xlabel("√(mask area) [px]"); ax[1].set_ylabel("objects")
plt.tight_layout(); savefig("fig_dataset_overview.png")

# --- 3 example objects per class (crop + mask): one column per class ---
N_EX = 3
rng = random.Random(SEED)
fig, axes = plt.subplots(N_EX, len(CLASSES), figsize=(3.1 * len(CLASSES), 3.0 * N_EX + 0.4))
for k, cname in enumerate(CLASSES):
    pool = [a for a in anns_all if a["category_id"] == k + 1 and a["area"] >= 48 ** 2] or \
           [a for a in anns_all if a["category_id"] == k + 1]
    picks, seen = [], set()
    for a in rng.sample(pool, len(pool)):
        if a["image_id"] not in seen:
            picks.append(a); seen.add(a["image_id"])
        if len(picks) == N_EX:
            break
    for j in range(N_EX):
        axj = axes[j, k]; axj.axis("off")
        if j >= len(picks):
            continue
        a = picks[j]; img = load_rgb(a["image_id"]); H, W = img.shape[:2]
        mk = poly_to_mask(a["segmentation"], H, W)
        x, y, w, h = a["bbox"]; pad = 0.25 * max(w, h) + 10
        x0, y0, x1, y1 = int(max(0, x - pad)), int(max(0, y - pad)), int(min(W, x + w + pad)), int(min(H, y + h + pad))
        axj.imshow(overlay(img, mk, CLASS_COLORS[k])[y0:y1, x0:x1])
        axj.set_title(a["taco_name"], fontsize=9, color=INK2)
plt.tight_layout(rect=(0, 0, 1, 0.94))
for k, cname in enumerate(CLASSES):                    # class name centred over each column
    bb = axes[0, k].get_position()
    fig.text((bb.x0 + bb.x1) / 2, 0.975, cname, ha="center", va="center", fontsize=13, fontweight="bold", color=CLASS_COLORS[k])
savefig("fig_class_examples.png")

## 3 · Convert to YOLO format (detection + segmentation labels)

In [ ]:
def polygon_area(p):
    xy = np.array(p).reshape(-1, 2)
    return 0.5 * abs(np.dot(xy[:, 0], np.roll(xy[:, 1], 1)) - np.dot(xy[:, 1], np.roll(xy[:, 0], 1)))

anns_by_img = {}
for a in anns_all:
    anns_by_img.setdefault(a["image_id"], []).append(a)

def write_yolo(root, seg):
    shutil.rmtree(root, ignore_errors=True)
    for s, ids in splits.items():
        (root / "images" / s).mkdir(parents=True); (root / "labels" / s).mkdir(parents=True)
        for i in ids:
            im = images_all[i]; W, H = im["width"], im["height"]
            src, dst = FAST_IMG / im["file_name"], root / "images" / s / im["file_name"]
            try:
                os.link(src, dst)                 # hard link: no extra disk space
            except OSError:
                shutil.copy2(src, dst)
            lines = []
            for a in anns_by_img.get(i, []):
                c = a["category_id"] - 1
                if seg:                            # YOLO-seg allows one polygon per object → keep the largest part
                    poly = max(a["segmentation"], key=polygon_area)
                    xy = np.clip(np.array(poly).reshape(-1, 2) / [W, H], 0, 1)
                    lines.append(f"{c} " + " ".join(f"{v:.6f}" for v in xy.ravel()))
                else:
                    x, y, w, h = a["bbox"]
                    lines.append(f"{c} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
            (root / "labels" / s / (Path(im["file_name"]).stem + ".txt")).write_text("\n".join(lines))
    cfg = {"path": str(root), "train": "images/train", "val": "images/val", "test": "images/test",
           "names": {k: n for k, n in enumerate(CLASSES)}}
    (root / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    return root / "data.yaml"

DET_YAML = write_yolo(LOCAL / "yolo_det", seg=False)     # local disk: fast to train from, rebuilt each session
SEG_YAML = write_yolo(LOCAL / "yolo_seg", seg=True)
print(DET_YAML.read_text())

## 4 · Train the YOLO detector (≈30 min on T4)
Fine-tunes a COCO-pretrained YOLO on the 5 litter classes. Its boxes become the prompts for EfficientSAM. EfficientSAM itself is **not** retrained (pretrained weights, zero-shot), as stated in the project scope.

In [ ]:
from ultralytics import YOLO

def train_yolo(weights, data_yaml, name, epochs, imgsz=None, batch=None):
    run = RUNS / name
    if RETRAIN and run.exists():
        print(f"RETRAIN = True → deleting old {name} weights in {run}")
        shutil.rmtree(run)
    best, last, done = run / "weights/best.pt", run / "weights/last.pt", run / "DONE"
    if done.exists() and best.exists():
        print(f"✓ {name} already trained → {best}")
        return best
    args = dict(data=str(data_yaml), epochs=epochs, imgsz=imgsz or IMG_SIZE, batch=batch or BATCH, patience=PATIENCE, seed=SEED,
                project=str(RUNS), name=name, exist_ok=True, cache=True, plots=True, device=YOLO_DEVICE, workers=2)
    if last.exists():
        try:
            print("↻ resuming from", last)
            YOLO(str(last)).train(resume=True)
        except Exception as e:                 # e.g. "training is finished, nothing to resume"
            print("resume skipped:", e)
            if not best.exists():
                YOLO(weights).train(**args)
    else:
        YOLO(weights).train(**args)
    done.touch()
    free_gpu()                                 # the trainer keeps GPU memory until it is garbage-collected
    return best

t0 = time.time()
DET_BEST = train_yolo(DET_MODEL, DET_YAML, "yolo_det", EPOCHS)
print(f"detector ready: {DET_BEST}  ({(time.time() - t0) / 60:.1f} min)")
for f in ["results.png", "BoxPR_curve.png", "confusion_matrix_normalized.png"]:
    if (RUNS / "yolo_det" / f).exists():
        shutil.copy(RUNS / "yolo_det" / f, FIG / f"det_train_{f}")

### 4.1 Train the YOLO-seg baseline (≈40 min, for comparison E4)

In [ ]:
SEG_BEST = None
if TRAIN_SEG_BASELINE:
    t0 = time.time()
    SEG_BEST = train_yolo(SEG_MODEL, SEG_YAML, "yolo_seg", SEG_EPOCHS)
    print(f"YOLO-seg ready: {SEG_BEST}  ({(time.time() - t0) / 60:.1f} min)")
    if (RUNS / "yolo_seg/results.png").exists():
        shutil.copy(RUNS / "yolo_seg/results.png", FIG / "seg_train_results.png")
else:
    print("TRAIN_SEG_BASELINE = False → skipping E4")

### 4.2 Training curves

In [ ]:
res = pd.read_csv(RUNS / "yolo_det/results.csv"); res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for col, lab, c in [("train/box_loss", "train box loss", "#2a78d6"), ("val/box_loss", "val box loss", "#eb6834"),
                    ("train/cls_loss", "train cls loss", "#1baf7a"), ("val/cls_loss", "val cls loss", "#eda100")]:
    if col in res:
        ax[0].plot(res["epoch"], res[col], lw=2, label=lab, color=c)
ax[0].set_title("Detector losses", loc="left"); ax[0].set_xlabel("epoch"); ax[0].legend(ncol=2, fontsize=8)
for col, lab, c in [("metrics/mAP50(B)", "val mAP50", "#2a78d6"), ("metrics/mAP50-95(B)", "val mAP50-95", "#eb6834")]:
    if col in res:
        ax[1].plot(res["epoch"], res[col], lw=2, label=lab, color=c)
ax[1].set_title("Detector validation mAP", loc="left"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, None); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig("fig_det_training_curves.png")

### 4.3 Detector v2: fix the confusion matrix (≈60–75 min on T4)
At the progress report the detector was the bottleneck: it found only 31% of the litter, and almost no **Glass** (recall 0.04) or **Other Waste** (0.06).
The confusion matrix showed that most errors were *missed objects* (true class → background), not class swaps. v2 targets the two causes:
* **Small objects** (43% of test objects are smaller than 32×32 px): train and predict at **`DET_V2_IMG_SIZE` = 960 px** instead of 640.
* **Class imbalance** (Plastic 47% of objects vs Glass 5%): **repeat-factor sampling** (LVIS, Gupta et al. 2019). Training images that contain a rare class are shown more often in each epoch: class *c* gets repeat factor r_c = max(1, √(t / f_c)), where f_c is the share of training images containing *c*.

v1 is kept unchanged as the "before" model, so every table and figure below compares **v1 (progress) vs v2 (final)** on the same test split.

In [ ]:
def rfs_yaml(data_yaml, t=RFS_T):
    # Repeat-factor sampling: write a train list in which images with rare classes appear several times
    root = Path(data_yaml).parent
    img_cls = {}
    for f in sorted((root / "labels/train").glob("*.txt")):
        img_cls[f.stem] = {int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()}
    n = len(img_cls)
    freq = Counter(c for cs in img_cls.values() for c in cs)
    r = {c: max(1.0, math.sqrt(t / (freq[c] / n))) for c in freq}
    lines, reps = [], Counter()
    for stem, cs in img_cls.items():
        k = max(1, round(max((r[c] for c in cs), default=1.0)))
        reps[k] += 1
        lines += [str(root / "images/train" / f"{stem}.jpg")] * k
    (root / "train_rfs.txt").write_text("\n".join(lines))
    cfg = yaml.safe_load(Path(data_yaml).read_text()); cfg["train"] = str(root / "train_rfs.txt")
    out = root / "data_rfs.yaml"; out.write_text(yaml.safe_dump(cfg, sort_keys=False))
    rfs_table = pd.DataFrame({"images with class (train)": [freq[k] for k in range(len(CLASSES))],
                              "share of images": [freq[k] / n for k in range(len(CLASSES))],
                              "repeat factor": [r[k] for k in range(len(CLASSES))]}, index=CLASSES).round(3)
    display(rfs_table); rfs_table.to_csv(OUT / "TABLE_E1b_repeat_factors.csv")
    print(f"train list: {len(lines)} entries for {n} images · images by repeat count: {dict(sorted(reps.items()))}")
    return out

DET_V1_BEST = DET_BEST                                   # progress-report detector (640 px, no rebalancing)
DET_V2_BEST = None
if TRAIN_DET_V2:
    v2_yaml = rfs_yaml(DET_YAML) if DET_V2_RFS else DET_YAML
    t0 = time.time()
    DET_V2_BEST = train_yolo(DET_V2_MODEL, v2_yaml, "yolo_det_v2", DET_V2_EPOCHS, imgsz=DET_V2_IMG_SIZE, batch=DET_V2_BATCH)
    print(f"detector v2 ready: {DET_V2_BEST}  ({(time.time() - t0) / 60:.1f} min)")
    for f in ["results.png", "BoxPR_curve.png", "confusion_matrix_normalized.png"]:
        if (RUNS / "yolo_det_v2" / f).exists():
            shutil.copy(RUNS / "yolo_det_v2" / f, FIG / f"det_v2_train_{f}")
else:
    print("TRAIN_DET_V2 = False → the final results use detector v1")

## 5 · E1: Detector evaluation on the **test** split (v1 = progress report, v2 = final)
Ultralytics metrics (P, R and mAP per class) for both detectors. The final pipeline below uses **v2**; section 5.1 adds imbalance-aware metrics, ROC/PR curves and before/after confusion matrices.

In [ ]:
def yolo_test_table(weights, imgsz, name):
    mdl = YOLO(str(weights))
    m = mdl.val(data=str(DET_YAML), split="test", imgsz=imgsz, batch=BATCH, device=YOLO_DEVICE,
                project=str(RUNS), name=name, exist_ok=True, plots=True, verbose=False)
    rows = []
    for j, c in enumerate(m.ap_class_index):
        p, r = float(m.box.p[j]), float(m.box.r[j])
        rows.append([CLASSES[c], int(stats.loc["test", CLASSES[c]]), p, r, 2 * p * r / (p + r + 1e-9), m.box.ap50[j], m.box.ap[j]])
    mp, mr = float(m.box.mp), float(m.box.mr)
    rows.append(["all", int(stats.loc["test", "objects"]), mp, mr, 2 * mp * mr / (mp + mr + 1e-9), m.box.map50, m.box.map])
    t = pd.DataFrame(rows, columns=["class", "GT objects", "Precision", "Recall", "F1", "mAP50", "mAP50-95"]).set_index("class")
    t[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]] = t[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]].astype(float).round(3)
    for f in (RUNS / name).glob("*.png"):
        shutil.copy(f, FIG / f"{name}_{f.name}")
    del mdl; free_gpu()
    return t

det_table_v1 = yolo_test_table(DET_V1_BEST, IMG_SIZE, "det_test")              # v1: same numbers as the progress report
det_table_v1.to_csv(OUT / "E1_detection_metrics.csv")
print("v1 (progress report detector):"); display(det_table_v1)
if DET_V2_BEST is not None:
    det_table_v2 = yolo_test_table(DET_V2_BEST, DET_V2_IMG_SIZE, "det_test_v2")
    det_table_v2.to_csv(OUT / "E1_detection_metrics_v2.csv")
    print("v2 (final detector):"); display(det_table_v2)
    e1_compare = pd.concat({"v1": det_table_v1, "v2": det_table_v2}, axis=1).swaplevel(axis=1)[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]]
    e1_compare.to_csv(OUT / "TABLE_E1_v1_vs_v2.csv"); display(e1_compare)
    DET_BEST, DET_IMGSZ, det_table = DET_V2_BEST, DET_V2_IMG_SIZE, det_table_v2
else:
    DET_BEST, DET_IMGSZ, det_table = DET_V1_BEST, IMG_SIZE, det_table_v1
det = YOLO(str(DET_BEST))                                # the detector used by everything below (E3–E7, prototype)
print(f"→ pipeline detector: {DET_BEST}  @ {DET_IMGSZ}px")

In [ ]:
# ---- predictions on the test images (kept for E3 and the figures) ----
with contextlib.redirect_stdout(io.StringIO()):
    test_coco = COCO(str(COCO_JSON["test"]))
test_ids = sorted(test_coco.getImgIds())
if MAX_TEST_IMAGES:
    test_ids = test_ids[:MAX_TEST_IMAGES]

def gt_of(i):
    anns = test_coco.loadAnns(test_coco.getAnnIds(imgIds=i))
    boxes = np.array([[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns],
                     np.float32).reshape(-1, 4)
    return dict(anns=anns, boxes=boxes, cls=np.array([a["category_id"] - 1 for a in anns], int),
                rles=[test_coco.annToRLE(a) for a in anns], area=np.array([a["area"] for a in anns], float))
GT = {i: gt_of(i) for i in test_ids}

det_preds = {}
paths = [str(FAST_IMG / images_all[i]["file_name"]) for i in test_ids]
# one image per call: Ultralytics turns a Python list of paths into ONE batch, which can run out of GPU memory
for i, pth in tqdm(list(zip(test_ids, paths)), desc="detector on test images"):
    r = det.predict(pth, conf=0.001, iou=0.7, max_det=100, imgsz=DET_IMGSZ, device=YOLO_DEVICE, verbose=False)[0]
    det_preds[i] = dict(boxes=r.boxes.xyxy.cpu().numpy().astype(np.float32), scores=r.boxes.conf.cpu().numpy(),
                        cls=r.boxes.cls.cpu().numpy().astype(int))

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

def match(pb, ps, gb, thr=MATCH_IOU):
    # Greedy class-agnostic matching, highest score first → [(pred_idx, gt_idx)], unmatched preds, unmatched GTs
    iou, used, pairs = box_iou(pb, gb), set(), []
    for p in np.argsort(-ps):
        if iou.shape[1] == 0:
            break
        cand = np.where((iou[p] >= thr) & ~np.isin(np.arange(iou.shape[1]), list(used)))[0]
        if len(cand):
            g = cand[np.argmax(iou[p, cand])]; used.add(g); pairs.append((int(p), int(g)))
    mp_ = {p for p, _ in pairs}
    return pairs, [p for p in range(len(pb)) if p not in mp_], [g for g in range(len(gb)) if g not in used]

# ---- confusion matrix + accuracy at CONF_THRES ----
nc = len(CLASSES); cm = np.zeros((nc + 1, nc + 1), int)      # rows = true (last = background), cols = predicted
per_image = {}
for i in test_ids:
    p, g = det_preds[i], GT[i]; k = p["scores"] >= CONF_THRES
    pb, ps, pc = p["boxes"][k], p["scores"][k], p["cls"][k]
    pairs, ump, umg = match(pb, ps, g["boxes"])
    for pi, gi in pairs: cm[g["cls"][gi], pc[pi]] += 1
    for pi in ump: cm[nc, pc[pi]] += 1
    for gi in umg: cm[g["cls"][gi], nc] += 1
    per_image[i] = dict(pairs=pairs, ump=ump, umg=umg, pb=pb, ps=ps, pc=pc,
                        n_ok=sum(pc[pi] == g["cls"][gi] for pi, gi in pairs), n_gt=len(g["boxes"]))
tp = np.diag(cm)[:nc]; fp = cm[:, :nc].sum(0) - tp; fn = cm[:nc, :].sum(1) - tp
P_, R_ = tp / np.maximum(tp + fp, 1), tp / np.maximum(tp + fn, 1)
thr_table = pd.DataFrame({"TP": tp, "FP": fp, "FN": fn, "Precision": P_, "Recall": R_,
                          "F1": 2 * P_ * R_ / np.maximum(P_ + R_, 1e-9)}, index=CLASSES)
matched = cm[:nc, :nc].sum()
cls_acc = np.trace(cm[:nc, :nc]) / max(matched, 1)
thr_table.loc["all (micro)"] = [tp.sum(), fp.sum(), fn.sum(), tp.sum() / max(tp.sum() + fp.sum(), 1),
                                tp.sum() / max(tp.sum() + fn.sum(), 1), 0]
thr_table.loc["all (micro)", "F1"] = 2 * thr_table.loc["all (micro)", "Precision"] * thr_table.loc["all (micro)", "Recall"] / max(
    thr_table.loc["all (micro)", "Precision"] + thr_table.loc["all (micro)", "Recall"], 1e-9)
thr_table = thr_table.round(3); thr_table[["TP", "FP", "FN"]] = thr_table[["TP", "FP", "FN"]].astype(int)
thr_table.to_csv(OUT / "E1_detection_at_conf_thres.csv")
print(f"At conf ≥ {CONF_THRES}, box IoU ≥ {MATCH_IOU}:  classification accuracy of localised objects = {cls_acc:.3f} "
      f"({np.trace(cm[:nc, :nc])}/{matched})")
display(thr_table)

labels = CLASSES + ["Background"]
norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("blue", ["#ffffff", "#cde2fb", "#6da7ec", "#256abf", "#0d366b"])
fig, ax = plt.subplots(figsize=(7.2, 6))
ax.imshow(norm, cmap=cmap, vmin=0, vmax=1); ax.grid(False)
for r_ in range(nc + 1):
    for c_ in range(nc + 1):
        if r_ == nc and c_ == nc:
            continue
        ax.text(c_, r_, f"{norm[r_, c_]:.2f}\n({cm[r_, c_]})", ha="center", va="center", fontsize=8,
                color="white" if norm[r_, c_] > 0.55 else INK)
ax.set_xticks(range(nc + 1), labels, rotation=30, ha="right"); ax.set_yticks(range(nc + 1), labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Detector {'v2' if DET_V2_BEST is not None else 'v1'} confusion matrix (test, conf ≥ {CONF_THRES}, row-normalised)", loc="left")
plt.tight_layout(); savefig("fig_confusion_matrix.png")

### 5.1 · E7 · Imbalance-aware metrics for the detector (answers "accuracy alone is not enough")
The test split is **imbalanced**: Plastic has about 7× more objects than Glass. A detector that is good only on Plastic
can still get a high accuracy. So here we report metrics that give **every class the same weight**, plus ROC / PR curves.
The code is a small reusable API, so the same functions evaluate **any** detector (v1 now, v2 later → before / after).

| Function | What it does |
|---|---|
| `load_gt(split)` | ground-truth boxes + classes for `"val"` or `"test"` |
| `predict_ids(model, ids)` | run a YOLO detector on these images (conf ≥ 0.001, one image per call) |
| `detection_report(preds, name, conf_thr=...)` | confusion matrix + per-class table + summary metrics (one global or 5 per-class thresholds) |
| `roc_pr_report({name: preds})` | object-level ROC-AUC and PR-AUC per class, with figures |
| `tune_class_thresholds(preds_val, gt_val, ids_val)` | one confidence threshold per class, chosen on **val** (never on test) |
| `compare_detectors({name: report})` | comparison table + before/after confusion matrices + F1 per class |

In [ ]:
from sklearn.metrics import (cohen_kappa_score, matthews_corrcoef, roc_auc_score, roc_curve,
                             average_precision_score, precision_recall_curve)
import re

NC = len(CLASSES)
E7_LINE_COLORS = ["#898781", "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # 1st detector grey = "before"


def _slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_")


def load_gt(split, max_images=None):
    """GT dict {img_id: dict(anns, boxes xyxy, cls, rles, area)} + sorted ids for 'val' or 'test' (same format as GT)."""
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(str(COCO_JSON[split]))
    ids = sorted(coco.getImgIds())
    if max_images:
        ids = ids[:max_images]
    gt = {}
    for i in ids:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=i))
        boxes = np.array([[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]]
                          for a in anns], np.float32).reshape(-1, 4)
        gt[i] = dict(anns=anns, boxes=boxes, cls=np.array([a["category_id"] - 1 for a in anns], int),
                     rles=[coco.annToRLE(a) for a in anns], area=np.array([a["area"] for a in anns], float))
    return gt, ids


def predict_ids(model, ids, imgsz=IMG_SIZE, desc="detector", **kw):
    """{img_id: dict(boxes, scores, cls)} like det_preds. Extra kwargs go to model.predict (e.g. augment=True)."""
    args = dict(conf=0.001, iou=0.7, max_det=100, imgsz=imgsz, device=YOLO_DEVICE, verbose=False)
    args.update(kw)
    preds = {}
    for i in tqdm(ids, desc=desc):             # one image per call (a list of paths = one big batch → GPU memory)
        r = model.predict(str(FAST_IMG / images_all[i]["file_name"]), **args)[0]
        preds[i] = dict(boxes=r.boxes.xyxy.cpu().numpy().astype(np.float32), scores=r.boxes.conf.cpu().numpy(),
                        cls=r.boxes.cls.cpu().numpy().astype(int))
    return preds


def _thr_vec(conf_thr):
    t = np.asarray(conf_thr, float)
    return np.full(NC, float(t)) if t.ndim == 0 else t.reshape(NC)


def _match_all(preds, gt, ids, conf_thr):
    """Greedy class-agnostic matching (cell 24 `match`) → 6×6 CM (last = background) + per-GT-object labels."""
    thr = _thr_vec(conf_thr)
    cm_ = np.zeros((NC + 1, NC + 1), int)
    y_true, y_pred = [], []                        # one entry per GT object; y_pred = NC means "missed"
    for i in ids:
        p, g = preds[i], gt[i]
        k = p["scores"] >= thr[p["cls"]] if len(p["cls"]) else np.zeros(0, bool)
        pb, ps, pc = p["boxes"][k], p["scores"][k], p["cls"][k]
        pairs, ump, umg = match(pb, ps, g["boxes"])
        lab = np.full(len(g["cls"]), NC)
        for pi, gi in pairs:
            cm_[g["cls"][gi], pc[pi]] += 1; lab[gi] = pc[pi]
        for pi in ump:
            cm_[NC, pc[pi]] += 1
        for gi in umg:
            cm_[g["cls"][gi], NC] += 1
        y_true.extend(g["cls"].tolist()); y_pred.extend(lab.tolist())
    return cm_, np.array(y_true, int), np.array(y_pred, int)


def _per_class(cm_):
    tp = np.diag(cm_)[:NC].astype(float)
    fp = cm_[:, :NC].sum(0) - tp
    support = cm_[:NC, :].sum(1).astype(float)     # every GT object of the class (matched or missed)
    fn = support - tp
    P = tp / np.maximum(tp + fp, 1); R = tp / np.maximum(support, 1)
    F = 2 * P * R / np.maximum(P + R, 1e-9)
    return tp, fp, fn, support, P, R, F


def detection_report(preds, name, gt=None, ids=None, conf_thr=CONF_THRES, detector=None, save=True, show=True):
    """conf_thr: float, or 5 per-class thresholds. Returns dict(name, detector, conf_thr, cm, per_class, summary, y_true, y_pred)."""
    gt = GT if gt is None else gt
    ids = test_ids if ids is None else ids
    thr = _thr_vec(conf_thr)
    cm_, y_true, y_pred = _match_all(preds, gt, ids, thr)
    tp, fp, fn, support, P, R, F = _per_class(cm_)
    per_class = pd.DataFrame({"conf_thr": thr, "support": support.astype(int), "TP": tp.astype(int), "FP": fp.astype(int),
                              "FN": fn.astype(int), "Precision": P, "Recall": R, "F1": F}, index=CLASSES)
    per_class.index.name = "class"
    has = support > 0                               # classes absent from the split (tiny dry runs) are left out of means
    mP, mR = tp.sum() / max(tp.sum() + fp.sum(), 1), tp.sum() / max(support.sum(), 1)
    matched = cm_[:NC, :NC].sum()
    summary = pd.Series({
        "Precision (micro)": mP, "Recall (micro)": mR, "F1 (micro)": 2 * mP * mR / max(mP + mR, 1e-9),
        "Precision (macro)": P[has].mean(), "Recall (macro)": R[has].mean(), "F1 (macro)": F[has].mean(),
        "F1 (weighted)": (F * support).sum() / max(support.sum(), 1),
        "Balanced accuracy": R[has].mean(),
        "Accuracy (all GT objects)": tp.sum() / max(support.sum(), 1),
        "Accuracy (localised objects)": np.trace(cm_[:NC, :NC]) / max(matched, 1),
        "Cohen kappa": cohen_kappa_score(y_true, y_pred, labels=list(range(NC + 1))) if len(y_true) else np.nan,
        "MCC": matthews_corrcoef(y_true, y_pred) if len(y_true) else np.nan,
        "GT objects": int(support.sum()), "Localised (IoU≥%.1f)" % MATCH_IOU: int(matched),
        "FP on background": int(cm_[NC, :NC].sum()),
    }, name=name)
    if save:
        per_class.round(3).to_csv(OUT / f"TABLE_E7_{_slug(name)}_per_class.csv")
        summary.round(4).to_frame().to_csv(OUT / f"TABLE_E7_{_slug(name)}_summary.csv")
    if show:
        print(f"── {name}  (conf_thr = {np.round(thr, 2).tolist()})")
        display(per_class.round(3)); display(summary.round(3).to_frame())
    return dict(name=name, detector=detector or name, conf_thr=thr, cm=cm_, per_class=per_class, summary=summary,
                y_true=y_true, y_pred=y_pred)

### What each summary metric means
We match predicted boxes to GT boxes (box IoU ≥ `MATCH_IOU`). **TP** = right box **and** right class. A GT object with no box, or with a box of the wrong class, is a **FN** for its class.
- **Precision (per class)** = TP / all boxes predicted as that class. "When the model says Glass, how often is it right?"
- **Recall (per class)** = TP / GT objects of that class. "How many of the Glass objects did we find with the right class?"
- **F1** = harmonic mean of precision and recall (high only if both are high).
- **micro** = add up TP/FP/FN of all classes first → dominated by the big class (Plastic).
- **macro** = compute per class, then take the simple mean → every class counts the same. **This is the main number for imbalanced data.**
- **weighted F1** = per-class F1 weighted by the number of GT objects (between micro and macro).
- **Balanced accuracy** = mean of per-class recall over **all** GT objects (a missed object counts as wrong). Same as macro recall.
- **Accuracy (all GT objects)** = correct objects / all GT objects (missed = wrong). Not balanced.
- **Accuracy (localised objects)** = the old v1 number: correct class among objects that **were found**. It hides the missed objects, so it looks too good.
- **Cohen's kappa** = agreement between true and predicted labels (predicted = matched class or "missed"), corrected for chance. 0 = chance, 1 = perfect.
- **MCC** (Matthews correlation coefficient) = correlation between true and predicted labels; uses the whole confusion table, robust to imbalance. 0 = chance, 1 = perfect.

Kappa and MCC are computed over GT objects, so false boxes on background are not in them (they are in Precision and "FP on background").

### Object-level ROC and PR curves (AUC)
A detector has no single "score per sample", so we build one per **GT object**: for each class *c*, `s_c` = the highest confidence of a class-*c* box whose IoU with the object is ≥ `MATCH_IOU` (0 if there is none). Ultralytics' NMS is per class, so overlapping boxes of different classes are kept → each object gets a 5-score vector.
Then, for each class, one-vs-rest **ROC-AUC** (sklearn `roc_auc_score`) and **PR-AUC** (average precision). **Macro** = mean over classes.
Objects that the detector **missed** get score 0 for every class, so low detector recall lowers the AUC (it is not hidden).
The straight line at the end of each curve = these missed objects (all tied at score 0).
False boxes on background have no GT object, so they do not enter the ROC; they appear in Precision instead.

In [ ]:
def object_scores(preds, gt=None, ids=None):
    """(N_gt_objects × 5) score matrix and the true class of every GT object."""
    gt = GT if gt is None else gt
    ids = test_ids if ids is None else ids
    S, y = [], []
    for i in ids:
        p, g = preds[i], gt[i]
        s = np.zeros((len(g["cls"]), NC))
        if len(g["cls"]) and len(p["cls"]):
            hit = np.where(box_iou(g["boxes"], p["boxes"]) >= MATCH_IOU, p["scores"][None, :], 0.0)   # G × P
            for c in range(NC):
                k = p["cls"] == c
                if k.any():
                    s[:, c] = hit[:, k].max(1)
        S.append(s); y.append(g["cls"])
    return np.concatenate(S) if S else np.zeros((0, NC)), np.concatenate(y) if y else np.zeros(0, int)


def roc_pr_report(preds_by_name, gt=None, ids=None):
    """preds_by_name: {detector name: preds}. Saves TABLE_E7_auc.csv, fig_E7_roc_curves.png, fig_E7_pr_curves.png."""
    rows, curves = [], {}
    for name, preds in preds_by_name.items():
        S, y = object_scores(preds, gt, ids)
        curves[name] = (S, y)
        for c in range(NC):
            pos = y == c
            ok = 0 < pos.sum() < len(y)
            rows.append([name, CLASSES[c], int(pos.sum()), roc_auc_score(pos, S[:, c]) if ok else np.nan,
                         average_precision_score(pos, S[:, c]) if ok else np.nan, pos.mean() if len(y) else np.nan])
        sub = pd.DataFrame(rows[-NC:])
        rows.append([name, "macro", int(len(y)), sub[3].mean(), sub[4].mean(), np.nan])
    auc = pd.DataFrame(rows, columns=["detector", "class", "GT objects", "ROC-AUC", "PR-AUC (AP)", "PR baseline (prevalence)"])
    auc = auc.set_index(["detector", "class"])
    auc.round(3).to_csv(OUT / "TABLE_E7_auc.csv")
    display(auc.round(3))

    n = len(curves)
    for kind in ("roc", "pr"):
        fig, axes = plt.subplots(1, n, figsize=(5.6 * n, 6.2), squeeze=False)
        for ax, (name, (S, y)) in zip(axes[0], curves.items()):
            for c in range(NC):
                pos = y == c
                if not 0 < pos.sum() < len(y):
                    continue
                a = auc.loc[(name, CLASSES[c])]
                if kind == "roc":
                    fpr, tpr, _ = roc_curve(pos, S[:, c])
                    ax.plot(fpr, tpr, lw=2, color=CLASS_COLORS[c], label=f"{CLASSES[c]}  AUC {a['ROC-AUC']:.3f}")
                else:
                    pr, rc, _ = precision_recall_curve(pos, S[:, c])
                    ax.plot(rc, pr, lw=2, color=CLASS_COLORS[c], label=f"{CLASSES[c]}  AP {a['PR-AUC (AP)']:.3f}")
                    ax.axhline(pos.mean(), color=CLASS_COLORS[c], lw=0.8, ls=":")
            m_ = auc.loc[(name, "macro")]
            if kind == "roc":
                ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1)
                ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
                ax.set_title(f"{name}\nmacro ROC-AUC {m_['ROC-AUC']:.3f}", loc="left")
            else:
                ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
                ax.set_title(f"{name}\nmacro AP {m_['PR-AUC (AP)']:.3f}  (dotted = chance level)", loc="left")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_aspect("equal"); ax.grid(axis="x")
            ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=2, columnspacing=1.2)
        fig.suptitle("Object-level " + ("ROC" if kind == "roc" else "PR") + " curves, one-vs-rest (test)",
                     x=0.01, ha="left", fontsize=11, color=INK2)
        plt.tight_layout(rect=(0, 0, 1, 0.96)); savefig(f"fig_E7_{kind}_curves.png")
    return auc

### Per-class confidence thresholds (tuned on **val**)
One global threshold may be unfair to some classes: the detector's confidence is not calibrated the same way for every class (e.g. it may be less confident on rare classes). Here every class gets its own threshold, chosen to **maximise that class's F1 on the validation split**. The test split is only used for the final report.
Search: (1) the same threshold for all classes over the grid → best value per class; (2) one round of coordinate search (change one class's threshold, keep the others fixed), because the class-agnostic matching makes classes interact a little.

In [ ]:
def tune_class_thresholds(preds_val, gt_val, ids_val, grid=np.arange(0.05, 0.91, 0.05)):
    """→ (5 per-class thresholds, table). Each class's threshold maximises its own F1 on val."""
    grid = np.round(np.asarray(grid, float), 3)
    f1_of = lambda t: _per_class(_match_all(preds_val, gt_val, ids_val, t)[0])[6]
    F_uni = np.stack([f1_of(t) for t in grid])                 # len(grid) × 5
    thr = grid[F_uni.argmax(0)].copy()
    for c in range(NC):                                          # 1 round of coordinate search
        best = (-1, thr[c])
        for t in grid:
            v = thr.copy(); v[c] = t
            f = f1_of(v)[c]
            if f > best[0] + 1e-12:
                best = (f, t)
        thr[c] = best[1]
    base = _per_class(_match_all(preds_val, gt_val, ids_val, CONF_THRES)[0])
    tuned = _per_class(_match_all(preds_val, gt_val, ids_val, thr)[0])
    table = pd.DataFrame({"val GT objects": base[3].astype(int), "threshold (uniform search)": grid[F_uni.argmax(0)],
                          "threshold (final)": thr, f"val F1 @ {CONF_THRES}": base[6], "val F1 @ tuned": tuned[6],
                          f"val Recall @ {CONF_THRES}": base[5], "val Recall @ tuned": tuned[5]}, index=CLASSES)
    table.index.name = "class"
    table.round(3).to_csv(OUT / "TABLE_E7_class_thresholds.csv")
    display(table.round(3))
    return thr, table

### Comparison table + before / after figures

In [ ]:
def compare_detectors(reports, auc=None, fig_suffix=""):
    """reports: {label: detection_report(...)}; auc (optional) = roc_pr_report table → adds macro ROC-AUC / AP by detector."""
    keys = ["F1 (macro)", "Balanced accuracy", "Precision (macro)", "Recall (macro)", "F1 (weighted)", "F1 (micro)",
            "Precision (micro)", "Recall (micro)", "Cohen kappa", "MCC", "Accuracy (all GT objects)",
            "Accuracy (localised objects)", "FP on background"]
    rows = {}
    for lab, r in reports.items():
        row = r["summary"][keys].copy()
        if auc is not None and (r["detector"], "macro") in auc.index:
            row["ROC-AUC (macro)"] = auc.loc[(r["detector"], "macro"), "ROC-AUC"]
            row["PR-AUC (macro)"] = auc.loc[(r["detector"], "macro"), "PR-AUC (AP)"]
        for c in CLASSES:
            row[f"F1 {c}"] = r["per_class"].loc[c, "F1"]
        for c in CLASSES:
            row[f"Recall {c}"] = r["per_class"].loc[c, "Recall"]
        rows[lab] = row
    table = pd.DataFrame(rows).T
    table.index.name = "detector / setting"
    table.round(3).to_csv(OUT / "TABLE_E7_detector_comparison.csv")
    display(table.round(3).T)

    # --- row-normalised confusion matrices side by side (same style as fig_confusion_matrix.png) ---
    labels = CLASSES + ["Background"]
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("blue", ["#ffffff", "#cde2fb", "#6da7ec", "#256abf", "#0d366b"])
    n = len(reports)
    fig, axes = plt.subplots(1, n, figsize=(5.9 * n + 0.6, 6.2), squeeze=False)
    for j, (ax, (lab, r)) in enumerate(zip(axes[0], reports.items())):
        cm_ = r["cm"]; norm = cm_ / np.maximum(cm_.sum(1, keepdims=True), 1)
        ax.imshow(norm, cmap=cmap, vmin=0, vmax=1); ax.grid(False)
        for a_ in range(NC + 1):
            for b_ in range(NC + 1):
                if a_ == NC and b_ == NC:
                    continue
                ax.text(b_, a_, f"{norm[a_, b_]:.2f}\n({cm_[a_, b_]})", ha="center", va="center", fontsize=8,
                        color="white" if norm[a_, b_] > 0.55 else INK)
        ax.set_xticks(range(NC + 1), labels, rotation=30, ha="right")
        ax.set_yticks(range(NC + 1), labels if j == 0 else [""] * (NC + 1))
        ax.set_xlabel("Predicted")
        if j == 0:
            ax.set_ylabel("True")
        s = r["summary"]
        ax.set_title(f"{lab}\nmacro-F1 {s['F1 (macro)']:.3f} · balanced acc {s['Balanced accuracy']:.3f}", loc="left", fontsize=11)
    fig.suptitle(f"Detector confusion matrices (test, row-normalised, box IoU ≥ {MATCH_IOU})", x=0.01, ha="left",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(rect=(0, 0, 1, 0.95)); savefig(f"fig_E7_confusion_before_after{fig_suffix}.png")

    # --- F1 per class, grouped bars, class support under the name ---
    support = next(iter(reports.values()))["per_class"]["support"]
    x = np.arange(NC); w = 0.8 / n
    fig, ax = plt.subplots(figsize=(max(9, 1.9 * NC + 1.2 * n), 4.4))
    for j, (lab, r) in enumerate(reports.items()):
        f = r["per_class"]["F1"].values
        b = ax.bar(x - 0.4 + w * (j + 0.5), f, w * 0.92, color=E7_LINE_COLORS[j % len(E7_LINE_COLORS)],
                   label=f"{lab}  (macro-F1 {r['summary']['F1 (macro)']:.3f})")
        for xi, v in zip(x - 0.4 + w * (j + 0.5), f):
            ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=7.5, color=INK2)
    ax.set_xticks(x, [f"{c}\n(n = {int(support[c])})" for c in CLASSES])
    ax.set_ylim(0, 1.05); ax.set_ylabel("F1 (right box and right class)")
    ax.set_title("Detector F1 per class on the test split (n = GT objects)", loc="left")
    ax.legend(fontsize=8.5, loc="upper right")
    plt.tight_layout(); savefig(f"fig_E7_f1_by_class{fig_suffix}.png")
    return table

### 5.2 Before / after: detector v1 (progress report) vs v2 (final)
Same test split, same matching rule (box IoU ≥ 0.5). For each detector, two operating points are reported: the global threshold `CONF_THRES` and per-class thresholds tuned on the **val** split (method 2 for the confusion matrix). ROC/PR curves are threshold-free.

In [ ]:
t0_e7 = time.time()
gt_val, val_ids = load_gt("val", max_images=MAX_TEST_IMAGES)

# ---- v1: re-predict with the progress-report detector (640 px) ----
det_v1 = YOLO(str(DET_V1_BEST))
preds_test_v1 = det_preds if DET_V2_BEST is None else predict_ids(det_v1, test_ids, imgsz=IMG_SIZE, desc="detector v1 on test")
preds_val_v1 = predict_ids(det_v1, val_ids, imgsz=IMG_SIZE, desc="detector v1 on val")
del det_v1; free_gpu()
rep = {f"v1 · conf {CONF_THRES}": detection_report(preds_test_v1, f"v1 · conf {CONF_THRES}", conf_thr=CONF_THRES, detector="v1")}
thr_v1, thr_table_v1 = tune_class_thresholds(preds_val_v1, gt_val, val_ids)
rep["v1 · per-class thr (val)"] = detection_report(preds_test_v1, "v1 · per-class thr (val)", conf_thr=thr_v1, detector="v1")
preds_by_det = {"v1": preds_test_v1}

# ---- v2: the pipeline detector (det_preds from section 5 were made with it) ----
if DET_V2_BEST is not None:
    preds_val_v2 = predict_ids(det, val_ids, imgsz=DET_IMGSZ, desc="detector v2 on val")
    rep[f"v2 · conf {CONF_THRES}"] = detection_report(det_preds, f"v2 · conf {CONF_THRES}", conf_thr=CONF_THRES, detector="v2")
    thr_v2, thr_table_v2 = tune_class_thresholds(preds_val_v2, gt_val, val_ids)
    rep["v2 · per-class thr (val)"] = detection_report(det_preds, "v2 · per-class thr (val)", conf_thr=thr_v2, detector="v2")
    preds_by_det["v2"] = det_preds
    chk = rep[f"v2 · conf {CONF_THRES}"]["per_class"][["TP", "FP", "FN"]].values == thr_table.loc[CLASSES, ["TP", "FP", "FN"]].values
    print("sanity check vs section 5 table:", "OK ✓" if chk.all() else "MISMATCH ✗")

auc_e7 = roc_pr_report(preds_by_det)
last = list(rep)[-2] if DET_V2_BEST is not None else list(rep)[0]          # v2 at the global threshold (or v1 alone)
first = f"v1 · conf {CONF_THRES}"
cmp_pair = compare_detectors({first: rep[first], last: rep[last]} if last != first else {first: rep[first]}, auc_e7)
cmp_e7 = compare_detectors(rep, auc_e7, fig_suffix="_all")                 # all 4 settings → TABLE_E7_detector_comparison.csv

# ---- findings (computed, not typed) ----
a, b = rep[first]["summary"], rep[last]["summary"]
pa, pb_ = rep[first]["per_class"], rep[last]["per_class"]
print(f"\nE7 findings ({int(a['GT objects'])} GT objects in {len(test_ids)} test images):")
print(f"• 'Accuracy of localised objects' (progress headline) hides the missed objects: v1 {a['Accuracy (localised objects)']:.3f} "
      f"but only {a['Accuracy (all GT objects)']:.3f} over all GT objects (balanced accuracy {a['Balanced accuracy']:.3f}).")
if last != first:
    print(f"• v1 → v2 at conf {CONF_THRES}: macro-F1 {a['F1 (macro)']:.3f} → {b['F1 (macro)']:.3f} · balanced accuracy "
          f"{a['Balanced accuracy']:.3f} → {b['Balanced accuracy']:.3f} · MCC {a['MCC']:.3f} → {b['MCC']:.3f} · "
          f"macro ROC-AUC {auc_e7.loc[('v1', 'macro'), 'ROC-AUC']:.3f} → {auc_e7.loc[('v2', 'macro'), 'ROC-AUC']:.3f}")
    print("• Recall per class v1 → v2: " + " · ".join(f"{c} {pa.loc[c, 'Recall']:.2f}→{pb_.loc[c, 'Recall']:.2f}" for c in CLASSES))
    t2 = rep["v2 · per-class thr (val)"]["summary"]
    print(f"• Per-class thresholds on v2: macro-F1 {b['F1 (macro)']:.3f} → {t2['F1 (macro)']:.3f}, balanced accuracy "
          f"{b['Balanced accuracy']:.3f} → {t2['Balanced accuracy']:.3f} (thresholds trade recall for precision; they cannot find missed objects).")
print(f"E7 done in {time.time() - t0_e7:.0f}s")

## 6 · EfficientSAM: load Ti and S, segment from box prompts
A box is passed to EfficientSAM as two points with labels **2 (top-left)** and **3 (bottom-right)**. The image is encoded once per image, then all boxes are decoded in batches. Of the 3 candidate masks, we keep the one with the highest predicted IoU.

In [ ]:
sys.path.insert(0, str(REPOS / "EfficientSAM"))
from efficient_sam.efficient_sam import build_efficient_sam

SAM_NAME = {"ti": "EfficientSAM-Ti", "s": "EfficientSAM-S"}

def load_esam(variant):
    wdir = REPOS / "EfficientSAM/weights"
    if variant == "s":
        if not (wdir / "efficient_sam_vits.pt").exists():
            with zipfile.ZipFile(wdir / "efficient_sam_vits.pt.zip") as z:
                z.extractall(wdir)
        dim, heads, ckpt = 384, 6, wdir / "efficient_sam_vits.pt"
    else:
        dim, heads, ckpt = 192, 3, wdir / "efficient_sam_vitt.pt"
    model = build_efficient_sam(encoder_patch_embed_dim=dim, encoder_num_heads=heads, checkpoint=None)
    model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=False)["model"])
    return model.eval().to(DEVICE)

ESAM = {v: load_esam(v) for v in SAM_VARIANTS}
PARAMS = {SAM_NAME[v]: sum(p.numel() for p in mdl.parameters()) / 1e6 for v, mdl in ESAM.items()}
PARAMS["YOLO detector"] = sum(p.numel() for p in det.model.parameters()) / 1e6
for k, v in PARAMS.items():
    print(f"{k:16s} {v:6.1f} M parameters")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def esam_embed(model, img):
    x = torch.from_numpy(img).permute(2, 0, 1).float().div(255).unsqueeze(0).to(DEVICE)
    return model.get_image_embeddings(x)

@torch.inference_mode()
def esam_decode(model, emb, boxes, H, W, chunk=32):
    # boxes: N×4 xyxy in pixels → N×H×W boolean masks
    out = []
    for k in range(0, len(boxes), chunk):
        b = torch.as_tensor(np.asarray(boxes[k:k + chunk]), dtype=torch.float32, device=DEVICE)
        n = b.shape[0]
        pts = b.view(1, n, 2, 2)
        lbl = torch.tensor([2, 3], device=DEVICE).view(1, 1, 2).expand(1, n, 2)
        logits, iou = model.predict_masks(emb, pts, lbl, multimask_output=True, input_h=H, input_w=W,
                                          output_h=-1, output_w=-1)
        low = logits[0, torch.arange(n, device=DEVICE), iou[0].argmax(-1)]          # n×256×256
        out.append((F.interpolate(low[:, None], (H, W), mode="bicubic")[:, 0] >= 0).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, H, W), bool)

def to_rle(mk):
    r = mask_utils.encode(np.asfortranarray(mk.astype(np.uint8)))
    r["counts"] = r["counts"].decode()
    return r

def rle_iou(a, b):
    return float(mask_utils.iou([a], [b], [0])[0, 0])

# warm-up (so the first timed image is not slower)
for v in SAM_VARIANTS:
    e = esam_embed(ESAM[v], np.zeros((480, 640, 3), np.uint8)); esam_decode(ESAM[v], e, np.array([[10, 10, 200, 200]]), 480, 640)
print("✓ EfficientSAM ready")

## 7 · E2 + E3: segmentation from GT boxes and from YOLO boxes
* **E2 (GT box prompts):** measures EfficientSAM alone, with a perfect box. IoU and Dice per object.
* **E3 (YOLO box prompts):** the real pipeline. COCO mask AP over all detections; IoU and Dice on detections matched to a GT object at `CONF_THRES`.

In [ ]:
free_gpu()
e2_rows, e3_rows, rles_e2 = [], [], {}
det_results, seg_results = [], {v: [] for v in SAM_VARIANTS}
enc_ms = {v: [] for v in SAM_VARIANTS}

def size_bin(a):
    return "small" if a < 32 ** 2 else ("medium" if a < 96 ** 2 else "large")

for i in tqdm(test_ids, desc="EfficientSAM on test images"):
    img = load_rgb(i); H, W = img.shape[:2]; g, p, pi_ = GT[i], det_preds[i], per_image[i]
    for b, s, c in zip(p["boxes"], p["scores"], p["cls"]):
        det_results.append({"image_id": i, "category_id": int(c) + 1, "score": float(s),
                            "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])]})
    keep_idx = np.where(p["scores"] >= CONF_THRES)[0]
    for v in SAM_VARIANTS:
        model = ESAM[v]
        sync(); t0 = time.perf_counter(); emb = esam_embed(model, img); sync(); enc_ms[v].append((time.perf_counter() - t0) * 1000)
        # E2: ground-truth boxes
        if len(g["boxes"]):
            for k, mk in enumerate(esam_decode(model, emb, g["boxes"], H, W)):
                r = to_rle(mk); iou = rle_iou(r, g["rles"][k]); rles_e2[(i, k, v)] = r
                e2_rows.append(dict(image_id=i, gt_idx=k, method=SAM_NAME[v], cls=CLASSES[g["cls"][k]], area=g["area"][k],
                                    size=size_bin(g["area"][k]), iou=iou, dice=2 * iou / (1 + iou)))
        # E3: YOLO boxes (all detections for mask AP; matched ones for IoU/Dice)
        if len(p["boxes"]):
            rl = [to_rle(mk) for mk in esam_decode(model, emb, p["boxes"], H, W)]
            for r, s, c in zip(rl, p["scores"], p["cls"]):
                seg_results[v].append({"image_id": i, "category_id": int(c) + 1, "segmentation": r, "score": float(s)})
            for pk, gk in pi_["pairs"]:
                j = keep_idx[pk]; iou = rle_iou(rl[j], g["rles"][gk])
                e3_rows.append(dict(image_id=i, gt_idx=gk, method=SAM_NAME[v], cls=CLASSES[g["cls"][gk]], size=size_bin(g["area"][gk]),
                                    iou=iou, dice=2 * iou / (1 + iou), box_iou=float(box_iou(p["boxes"][j:j + 1], g["boxes"][gk:gk + 1])[0, 0]),
                                    class_ok=bool(p["cls"][j] == g["cls"][gk])))
e2 = pd.DataFrame(e2_rows); e3 = pd.DataFrame(e3_rows)
e2.to_csv(OUT / "E2_gt_box_per_object.csv", index=False); e3.to_csv(OUT / "E3_yolo_box_matched_per_object.csv", index=False)
print(f"E2: {len(e2)} object masks · E3: {len(det_results)} detections, {len(e3)} matched masks")

## 7.1 · E6: effect of the prompt on EfficientSAM
**Question.** How much does the *type* and the *quality* of the prompt change the mask EfficientSAM returns, and can a better prompt fix the thin / elongated objects (ropes, straps) where a box makes EfficientSAM fill the whole box?
**Prompts** (each test image is encoded once per model; every prompt reuses that embedding):
* *Practical* (no ground truth needed): **P1** GT box (= E2) · **P2** one positive point at the box centre · **P3** box + box-centre point · and the real pipeline prompt, the **YOLO box** (± its centre point), on the objects the detector found.
* *Oracle "user click"* (upper bound of what a human click adds): **P4** one point at the most interior pixel of the GT mask (distance-transform maximum) · **P5** box + that point · **P6** box + 3 interior points spread out by farthest-point sampling.
* *Box quality*: GT boxes whose 4 sides are moved randomly by ±5 / 10 / 20 % of the box width/height (fixed seed), alone and + their centre point: how fast does the mask degrade when the detector's box is imprecise?
Metrics per object: mask IoU, Dice, IoU ≥ 0.5 and ≥ 0.75 rates, split by class, size and **fill ratio** = mask area / box area (low fill = thin, diagonal or hollow object).

In [ ]:
# ---- E6 setup: prompt definitions + helpers ----
free_gpu()
E6_PROMPTS = {   # key → (label, group)
    "P1": ("GT box", "practical"),
    "P2": ("Box-centre point", "practical"),
    "P3": ("GT box + centre point", "practical"),
    "P4": ("Oracle interior point", "oracle"),
    "P5": ("GT box + oracle point", "oracle"),
    "P6": ("GT box + 3 oracle points", "oracle"),
    "J05": ("GT box ±5% jitter", "box error"),
    "J10": ("GT box ±10% jitter", "box error"),
    "J20": ("GT box ±20% jitter", "box error"),
    "J05c": ("GT box ±5% jitter + centre point", "box error"),
    "J10c": ("GT box ±10% jitter + centre point", "box error"),
    "J20c": ("GT box ±20% jitter + centre point", "box error"),
    "Y1": ("YOLO box", "pipeline"),
    "Y3": ("YOLO box + centre point", "pipeline"),
}
E6_JITTER = {"J05": 0.05, "J10": 0.10, "J20": 0.20}
E6_MAIN = ["P1", "P2", "P3", "P4", "P5", "P6"]
E6_TAG = {v: {"ti": "Ti", "s": "S"}.get(v, v) for v in SAM_VARIANTS}
E6_PROMPT_COLORS = dict(zip(E6_MAIN, ["#2a78d6", "#a9c8ef", "#0d366b", "#f2b290", "#eb6834", "#a3361a"]))
FILL_BINS, FILL_LABELS = [0, 0.3, 0.6, 1.01], ["thin (<0.3)", "mid (0.3–0.6)", "compact (≥0.6)"]

def e6_box(b):
    return np.array([[b[0], b[1]], [b[2], b[3]]], np.float32), np.array([2, 3], np.int64)

def e6_pts(p):
    p = np.asarray(p, np.float32).reshape(-1, 2)
    return p, np.ones(len(p), np.int64)

def e6_cat(*ps):
    return np.concatenate([p[0] for p in ps]), np.concatenate([p[1] for p in ps])

def e6_centre(b):
    return [[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]]

def interior_points(mk, k=3, max_cand=4000):
    # k positive "clicks" inside the GT mask: the most interior pixel (distance-transform max), then farthest-point
    # sampling over the mask core (pixels at ≥30% of the max distance to the border) so the points are spread out
    if not mk.any():
        return None
    dt = cv2.distanceTransform(np.pad(mk.astype(np.uint8), 1), cv2.DIST_L2, 5)[1:-1, 1:-1]
    y0, x0 = np.unravel_index(int(dt.argmax()), dt.shape)
    ys, xs = np.nonzero(dt >= 0.3 * dt.max())
    if len(xs) > max_cand:
        sel = np.linspace(0, len(xs) - 1, max_cand).astype(int); ys, xs = ys[sel], xs[sel]
    pts, d = [(x0, y0)], np.hypot(xs - x0, ys - y0)
    for _ in range(k - 1):
        j = int(d.argmax()); pts.append((xs[j], ys[j])); d = np.minimum(d, np.hypot(xs - xs[j], ys - ys[j]))
    return np.array(pts, np.float32)

def jitter_box(b, s, rng, H, W):
    w, h = b[2] - b[0], b[3] - b[1]
    nb = b + rng.uniform(-s, s, 4) * np.array([w, h, w, h])
    nb = np.clip(nb, 0, [W - 1, H - 1, W - 1, H - 1])
    nb[2] = max(nb[2], nb[0] + 1); nb[3] = max(nb[3], nb[1] + 1)
    return nb.astype(np.float32)

@torch.inference_mode()
def esam_decode_prompts(model, emb, prompts, H, W, gt_masks=None, gt_idx=None, chunk=16, keep_masks=False):
    """Decode mixed point/box prompts in chunks.
    prompts: list of (points K×2 in px, labels K) — 1 = positive point, 2/3 = box corners; shorter prompts are padded with label -1.
    Returns pred_iou (n×3) for the 3 candidate masks, mask IoU vs gt_masks[gt_idx] (n×3, if given) and, with keep_masks,
    the candidate chosen by predicted IoU (same rule as esam_decode) as N×H×W bool."""
    piou_all, iou_all, masks = [], [], []
    for k in range(0, len(prompts), chunk):
        part = prompts[k:k + chunk]; m = len(part); K = max(len(l) for _, l in part)
        pts = np.full((m, K, 2), -1, np.float32); lbl = np.full((m, K), -1, np.int64)
        for j, (p, l) in enumerate(part):
            pts[j, :len(l)] = p; lbl[j, :len(l)] = l
        logits, piou = model.predict_masks(emb, torch.from_numpy(pts).to(DEVICE)[None], torch.from_numpy(lbl).to(DEVICE)[None],
                                           multimask_output=True, input_h=H, input_w=W, output_h=-1, output_w=-1)
        logits, piou = logits[0], piou[0]                                            # m×3×256×256, m×3
        piou_all.append(piou.float().cpu().numpy())
        if gt_masks is not None:
            mk = F.interpolate(logits.reshape(m * 3, 1, *logits.shape[-2:]), (H, W), mode="bicubic").view(m, 3, H, W) >= 0
            g = gt_masks[torch.as_tensor(np.asarray(gt_idx[k:k + chunk]), device=DEVICE)][:, None]
            inter = (mk & g).sum((-1, -2)).float(); uni = (mk | g).sum((-1, -2)).float()
            iou_all.append((inter / uni.clamp(min=1)).cpu().numpy()); del mk, g
        if keep_masks:
            low = logits[torch.arange(m, device=DEVICE), piou.argmax(-1)]
            masks.append((F.interpolate(low[:, None], (H, W), mode="bicubic")[:, 0] >= 0).cpu().numpy())
    cat = lambda xs, shape: np.concatenate(xs) if xs else np.zeros(shape)
    return (cat(piou_all, (0, 3)), cat(iou_all, (0, 3)) if gt_masks is not None else None,
            cat(masks, (0, H, W)).astype(bool) if keep_masks else None)

def e6_build_prompts(i, H, W):
    # every prompt of one image (model independent) → list of (key, gt_idx, (pts, lbl), prompt box or None)
    g, out = GT[i], []
    for k, b in enumerate(g["boxes"]):
        mk = mask_utils.decode(g["rles"][k]).astype(bool)
        ip = interior_points(mk, 3)
        if ip is None:                                     # degenerate annotation → fall back to the box centre
            ip = np.repeat(np.array(e6_centre(b), np.float32), 3, 0)
        out += [("P1", k, e6_box(b), b), ("P2", k, e6_pts(e6_centre(b)), None),
                ("P3", k, e6_cat(e6_box(b), e6_pts(e6_centre(b))), b),
                ("P4", k, e6_pts(ip[:1]), None), ("P5", k, e6_cat(e6_box(b), e6_pts(ip[:1])), b),
                ("P6", k, e6_cat(e6_box(b), e6_pts(ip)), b)]
        for key, s in E6_JITTER.items():
            rng = np.random.default_rng([SEED, int(i), k, int(round(s * 100))])
            jb = jitter_box(b, s, rng, H, W)
            out += [(key, k, e6_box(jb), jb), (key + "c", k, e6_cat(e6_box(jb), e6_pts(e6_centre(jb))), jb)]
    pim = per_image[i]
    for pk, gk in pim["pairs"]:                            # the real pipeline prompt (YOLO box at CONF_THRES, matched objects)
        yb = pim["pb"][pk].astype(np.float32)
        out += [("Y1", gk, e6_box(yb), yb), ("Y3", gk, e6_cat(e6_box(yb), e6_pts(e6_centre(yb))), yb)]
    return out

In [ ]:
# ---- E6 run: every GT object of the test split × every prompt × Ti and S ----
free_gpu()
e6_rows, E6_PROMPT_CACHE = [], {}
e6_sec = {v: 0.0 for v in SAM_VARIANTS}
t_all = time.perf_counter()
for i in tqdm(test_ids, desc="E6 prompt study"):
    g = GT[i]
    if len(g["boxes"]) == 0:
        continue
    img = load_rgb(i); H, W = img.shape[:2]
    plist = e6_build_prompts(i, H, W)
    for key, k, pr, _ in plist:
        E6_PROMPT_CACHE[(i, k, key)] = pr
    gt_np = np.stack([mask_utils.decode(r).astype(bool) for r in g["rles"]])
    fill = np.array([mask_utils.area(r) for r in g["rles"]], float) / np.maximum(
        (g["boxes"][:, 2] - g["boxes"][:, 0]) * (g["boxes"][:, 3] - g["boxes"][:, 1]), 1)
    pbox_iou = [float(box_iou(pb[None], g["boxes"][k:k + 1])[0, 0]) if pb is not None else np.nan for _, k, _, pb in plist]
    for v in SAM_VARIANTS:
        sync(); t0 = time.perf_counter()
        gt_t = torch.from_numpy(gt_np).to(DEVICE)
        emb = esam_embed(ESAM[v], img)
        piou, miou, _ = esam_decode_prompts(ESAM[v], emb, [p[2] for p in plist], H, W, gt_t, [p[1] for p in plist])
        del emb, gt_t; sync(); e6_sec[v] += time.perf_counter() - t0
        best = piou.argmax(1)
        for j, (key, k, _, _) in enumerate(plist):
            iou = float(miou[j, best[j]])
            e6_rows.append(dict(image_id=i, gt_idx=k, method=SAM_NAME[v], prompt=key, prompt_name=E6_PROMPTS[key][0],
                                group=E6_PROMPTS[key][1], cls=CLASSES[g["cls"][k]], area=g["area"][k], size=size_bin(g["area"][k]),
                                fill=float(fill[k]), iou=iou, dice=2 * iou / (1 + iou), out_idx=int(best[j]),
                                pred_iou=float(piou[j, best[j]]), iou_m0=float(miou[j, 0]), iou_m1=float(miou[j, 1]),
                                iou_m2=float(miou[j, 2]), prompt_box_iou=pbox_iou[j]))
e6 = pd.DataFrame(e6_rows)
e6["fill_bin"] = pd.cut(e6["fill"], FILL_BINS, labels=FILL_LABELS, right=False).astype(str)
e6.to_csv(OUT / "E6_prompt_per_object.csv", index=False)
free_gpu()
t_all = time.perf_counter() - t_all
n_obj = e6[e6.prompt == "P1"].groupby("method").size().max() if len(e6) else 0
print(f"E6: {len(e6)} rows · {n_obj} GT objects × {len(E6_PROMPTS)} prompt types · {len(SAM_VARIANTS)} models · "
      f"wall time {t_all / 60:.1f} min (" + ", ".join(f"{SAM_NAME[v]} {e6_sec[v]:.0f}s GPU" for v in SAM_VARIANTS) + ")")

# sanity checks: P1 must reproduce E2 (same box prompt), Y1 must reproduce E3 (same YOLO boxes)
chk = e6[e6.prompt == "P1"].merge(e2, on=["image_id", "gt_idx", "method"], suffixes=("", "_e2"))
print(f"sanity P1 vs E2: {len(chk)} objects, max |ΔIoU| = {np.abs(chk.iou - chk.iou_e2).max():.2e}")
if len(e3):
    chk = e6[e6.prompt == "Y1"].merge(e3, on=["image_id", "gt_idx", "method"], suffixes=("", "_e3"))
    print(f"sanity Y1 vs E3: {len(chk)} objects, max |ΔIoU| = {np.abs(chk.iou - chk.iou_e3).max():.2e}")

In [ ]:
# ---- E6 tables ----
def e6_summ(d):
    return {"mIoU": d.iou.mean(), "mDice": d.dice.mean(), "IoU≥0.5": (d.iou >= 0.5).mean(), "IoU≥0.75": (d.iou >= 0.75).mean()}

E6_MATCHED = set(map(tuple, e6.loc[e6.prompt == "Y1", ["image_id", "gt_idx"]].drop_duplicates().values))
e6["yolo_matched"] = [(a, b) in E6_MATCHED for a, b in zip(e6.image_id, e6.gt_idx)]

def e6_block(df, keys, subset):
    rows = []
    for key in keys:
        r = {"prompt": E6_PROMPTS[key][0], "key": key, "group": E6_PROMPTS[key][1], "objects": subset}
        for v in SAM_VARIANTS:
            d = df[(df.prompt == key) & (df.method == SAM_NAME[v])]
            r.update({f"{m} {E6_TAG[v]}": x for m, x in e6_summ(d).items()})
        r["n objects"] = len(d)
        rows.append(r)
    return rows

E6_ALL_KEYS = E6_MAIN + ["J05", "J10", "J20", "J05c", "J10c", "J20c"]
E6_SUB_KEYS = ["P1", "P3", "P5", "Y1", "Y3"]
e6_overall = pd.DataFrame(e6_block(e6, E6_ALL_KEYS, "all GT objects")
                          + e6_block(e6[e6.yolo_matched], E6_SUB_KEYS, "YOLO-matched objects"))
e6_overall.round(3).to_csv(OUT / "TABLE_E6_prompt_overall.csv", index=False)
display(e6_overall.round(3))

def e6_by(col, order):
    d = e6[e6.prompt.isin(E6_MAIN)]
    t = d.pivot_table(index=["method", "prompt"], columns=col, values="iou", aggfunc="mean")
    order = [o for o in order if o in t.columns]
    t = t[order]
    n = d[(d.prompt == "P1") & (d.method == SAM_NAME[SAM_VARIANTS[0]])][col].value_counts()
    t.columns = [f"{c} (n={int(n.get(c, 0))})" for c in order]
    t = t.reindex([(SAM_NAME[v], k) for v in SAM_VARIANTS for k in E6_MAIN])
    t.insert(0, "prompt_name", [E6_PROMPTS[k][0] for _, k in t.index])
    return t

e6_cls = e6_by("cls", CLASSES); e6_size = e6_by("size", ["small", "medium", "large"]); e6_fill = e6_by("fill_bin", FILL_LABELS)
e6_cls.round(3).to_csv(OUT / "TABLE_E6_prompt_by_class.csv")
e6_size.round(3).to_csv(OUT / "TABLE_E6_prompt_by_size.csv")
e6_fill.round(3).to_csv(OUT / "TABLE_E6_prompt_by_fill.csv")
display(e6_fill.round(3)); display(e6_size.round(3))

# box-quality robustness: IoU vs jitter level (0 % = exact GT box), box alone and box + centre point
jrows = []
for lvl, kb, kc in [(0, "P1", "P3"), (5, "J05", "J05c"), (10, "J10", "J10c"), (20, "J20", "J20c")]:
    r = {"box prompt": f"GT box ±{lvl}%", "jitter %": lvl, "objects": "all GT objects",
         "mean box IoU vs GT": e6[(e6.prompt == kb)].prompt_box_iou.mean()}
    for v in SAM_VARIANTS:
        r[f"mIoU box {E6_TAG[v]}"] = e6[(e6.prompt == kb) & (e6.method == SAM_NAME[v])].iou.mean()
        r[f"mIoU box+centre {E6_TAG[v]}"] = e6[(e6.prompt == kc) & (e6.method == SAM_NAME[v])].iou.mean()
    r["n objects"] = int((e6.prompt == kb).sum() / len(SAM_VARIANTS)); jrows.append(r)
if E6_MATCHED:
    r = {"box prompt": "YOLO box (matched)", "jitter %": np.nan, "objects": "YOLO-matched objects",
         "mean box IoU vs GT": e6[e6.prompt == "Y1"].prompt_box_iou.mean()}
    for v in SAM_VARIANTS:
        r[f"mIoU box {E6_TAG[v]}"] = e6[(e6.prompt == "Y1") & (e6.method == SAM_NAME[v])].iou.mean()
        r[f"mIoU box+centre {E6_TAG[v]}"] = e6[(e6.prompt == "Y3") & (e6.method == SAM_NAME[v])].iou.mean()
    r["n objects"] = len(E6_MATCHED); jrows.append(r)
e6_jit = pd.DataFrame(jrows)
e6_jit.round(3).to_csv(OUT / "TABLE_E6_box_jitter.csv", index=False)
display(e6_jit.round(3))

# multimask output selection: predicted-IoU pick (used everywhere) vs a fixed output index vs oracle best of 3
mrows = []
for key in E6_MAIN:
    for v in SAM_VARIANTS:
        d = e6[(e6.prompt == key) & (e6.method == SAM_NAME[v])]
        mrows.append({"prompt": E6_PROMPTS[key][0], "model": SAM_NAME[v], "pick by predicted IoU": d.iou.mean(),
                      "always output 0": d.iou_m0.mean(), "always output 1": d.iou_m1.mean(), "always output 2": d.iou_m2.mean(),
                      "oracle best of 3": d[["iou_m0", "iou_m1", "iou_m2"]].max(axis=1).mean()})
e6_mm = pd.DataFrame(mrows)
e6_mm.round(3).to_csv(OUT / "TABLE_E6_multimask_selection.csv", index=False)
display(e6_mm.round(3))

In [ ]:
# ---- E6 figures ----
sam_methods = [SAM_NAME[v] for v in SAM_VARIANTS]
ov_all = e6_overall[e6_overall.objects == "all GT objects"].set_index("key")
ov_sub = e6_overall[e6_overall.objects == "YOLO-matched objects"].set_index("key")

def e6_bars(ax, keys, tab, title):
    x = np.arange(len(keys)); w = 0.8 / len(SAM_VARIANTS)
    for j, v in enumerate(SAM_VARIANTS):
        vals = tab.loc[keys, f"mIoU {E6_TAG[v]}"].values.astype(float)
        bars = ax.bar(x + (j - (len(SAM_VARIANTS) - 1) / 2) * w, vals, w * 0.92, label=SAM_NAME[v], color=METHOD_COLORS[SAM_NAME[v]])
        for b_, v_ in zip(bars, vals):
            ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.01, f"{v_:.3f}", ha="center", va="bottom", fontsize=7, color=INK2)
    ax.set_xticks(x, [E6_PROMPTS[k][0].replace(" + ", "\n+ ") for k in keys], fontsize=8)
    ax.set_ylim(0, 1.25); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel("mean mask IoU"); ax.set_title(title, loc="left")
    ax.legend(fontsize=8, ncol=len(SAM_VARIANTS), loc="upper left")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={"width_ratios": [6, 4.2]})
e6_bars(ax[0], E6_MAIN, ov_all, f"E6 · Prompt type vs mask IoU (all {int(ov_all['n objects'].iloc[0])} GT objects)")
ax[0].axvspan(2.5, 5.5, color="#f4f3ee", zorder=0); ax[0].text(4, 1.03, "oracle clicks (upper bound)", ha="center", fontsize=8, color=MUTED)
ax[0].text(1, 1.03, "practical (no GT needed)", ha="center", fontsize=8, color=MUTED)
if len(ov_sub):
    e6_bars(ax[1], E6_SUB_KEYS, ov_sub, f"Same objects, real YOLO boxes (n={int(ov_sub['n objects'].iloc[0])})")
else:
    ax[1].axis("off")
plt.tight_layout(); savefig("fig_E6_prompt_overall.png")

fig, axes = plt.subplots(len(SAM_VARIANTS), 2, figsize=(13, 3.6 * len(SAM_VARIANTS)), squeeze=False)
for r_, v in enumerate(SAM_VARIANTS):
    for c_, (tab, name) in enumerate([(e6_fill, "fill ratio (mask area / box area)"), (e6_size, "object size")]):
        a_ = axes[r_, c_]; t = tab.loc[SAM_NAME[v]].drop(columns="prompt_name")
        x = np.arange(t.shape[1]); w = 0.8 / len(E6_MAIN)
        for j, k in enumerate(E6_MAIN):
            a_.bar(x + (j - (len(E6_MAIN) - 1) / 2) * w, t.loc[k].values.astype(float), w * 0.92, color=E6_PROMPT_COLORS[k], label=E6_PROMPTS[k][0])
        a_.set_xticks(x, [c.replace(" (", "\n(") for c in t.columns]); a_.set_ylim(0, 1.2); a_.set_yticks(np.arange(0, 1.01, 0.2))
        a_.set_ylabel("mean mask IoU"); a_.set_title(f"{SAM_NAME[v]} · IoU by {name}", loc="left")
        if r_ == 0 and c_ == 0:
            a_.legend(fontsize=7.5, ncol=3, loc="upper left")
plt.tight_layout(); savefig("fig_E6_prompt_by_fill_or_size.png")

fig, ax = plt.subplots(figsize=(7.2, 4))
jj = e6_jit[e6_jit["jitter %"].notna()]
for v in SAM_VARIANTS:
    c = METHOD_COLORS[SAM_NAME[v]]
    ax.plot(jj["jitter %"], jj[f"mIoU box {E6_TAG[v]}"], "-o", color=c, label=f"{SAM_NAME[v]} · box")
    ax.plot(jj["jitter %"], jj[f"mIoU box+centre {E6_TAG[v]}"], "--s", color=c, alpha=0.75, label=f"{SAM_NAME[v]} · box + centre point")
ax.set_xticks(jj["jitter %"], [f"±{int(l)}%\n(box IoU {b:.2f})" for l, b in zip(jj["jitter %"], jj["mean box IoU vs GT"])])
ax.set_xlabel("random shift of each box side (% of box width / height)"); ax.set_ylabel("mean mask IoU"); ax.grid(axis="both")
ax.set_title("E6 · Robustness of EfficientSAM to box error", loc="left"); ax.legend(fontsize=7.5)
plt.tight_layout(); savefig("fig_E6_box_jitter.png")

In [ ]:
# ---- E6 qualitative: thin / hard objects, box vs point prompts ----
def e6_crop_box(b, H, W, pad=0.3):          # same crop rule as crop_box in the qualitative-results section
    x0, y0, x1, y1 = b; w, h = x1 - x0, y1 - y0; p = pad * max(w, h) + 8
    return int(max(0, x0 - p)), int(max(0, y0 - p)), int(min(W, x1 + p)), int(min(H, y1 + p))

ref_v = SAM_VARIANTS[-1]; ref = SAM_NAME[ref_v]
p1 = e6[(e6.method == ref) & (e6.prompt == "P1")]
cand = p1[(p1.area >= 40 ** 2) & (p1.fill < 0.4)].sort_values("iou")
if len(cand) < 4:
    cand = p1[p1.area >= 40 ** 2].sort_values("iou")
picks, seen = [], set()
for rw in cand.itertuples():                       # worst box-prompt masks, at most one object per image
    if rw.image_id not in seen:
        picks.append(rw); seen.add(rw.image_id)
    if len(picks) == 4:
        break
E6_SHOW = [("P1", "Box"), ("P4", "Point"), ("P5", "Box + point"), ("P6", "Box + 3 points")]
if picks:
    fig, axes = plt.subplots(len(picks), 1 + len(E6_SHOW), figsize=(2.4 * (1 + len(E6_SHOW)), 2.45 * len(picks)), squeeze=False)
    for r_, rw in enumerate(picks):
        i, k = rw.image_id, rw.gt_idx
        img = load_rgb(i); H, W = img.shape[:2]; g = GT[i]
        x0, y0, x1, y1 = e6_crop_box(g["boxes"][k], H, W)
        th = max(2, int(0.006 * max(x1 - x0, y1 - y0))); rad = max(4, int(0.025 * max(x1 - x0, y1 - y0)))
        emb = esam_embed(ESAM[ref_v], img)
        _, _, mks = esam_decode_prompts(ESAM[ref_v], emb, [E6_PROMPT_CACHE[(i, k, key)] for key, _ in E6_SHOW], H, W, keep_masks=True)
        del emb
        gt_mask = mask_utils.decode(g["rles"][k]).astype(bool)
        panels = [("GT", None, gt_mask)] + [(lab, key, mk) for (key, lab), mk in zip(E6_SHOW, mks)]
        for c_, (lab, key, mk) in enumerate(panels):
            a_ = axes[r_, c_]; a_.axis("off")
            vis = overlay(img, mk, "#1baf7a" if key is None else METHOD_COLORS[ref], thick=th)
            if key in (None, "P1", "P5", "P6"):
                bx = g["boxes"][k].astype(int); cv2.rectangle(vis, tuple(bx[:2]), tuple(bx[2:]), (255, 255, 255), th)
            if key is not None:
                pts, lbl = E6_PROMPT_CACHE[(i, k, key)]
                for (px, py), l in zip(pts, lbl):
                    if l == 1:
                        cv2.circle(vis, (int(px), int(py)), rad, (255, 255, 255), -1, cv2.LINE_AA)
                        cv2.circle(vis, (int(px), int(py)), max(2, rad - th), (27, 175, 122), -1, cv2.LINE_AA)
            a_.imshow(vis[y0:y1, x0:x1])
            if key is None:
                a_.set_title(f"GT · {rw.cls}\nfill {rw.fill:.2f}", fontsize=8, color=INK)
            else:
                iou = e6[(e6.image_id == i) & (e6.gt_idx == k) & (e6.method == ref) & (e6.prompt == key)].iou.iloc[0]
                a_.set_title(f"{lab}\nIoU {iou:.2f}", fontsize=8, color=INK2)
    plt.suptitle(f"E6 · {ref}: hardest thin objects for the box prompt, and what oracle clicks add (green dots = positive points)",
                 fontweight="bold", y=1.01)
    plt.tight_layout(); savefig("fig_E6_prompt_examples.png")
free_gpu()

### E6 findings
The bullets below are computed from the E6 tables (all numbers are printed, not typed). *Practical* prompts can be used by the real pipeline; *oracle* points use the GT mask and are an upper bound for "a user clicks on the object" or for a future point generator.

In [ ]:
def _m(key, v, sub=False):
    t = ov_sub if sub else ov_all
    return float(t.loc[key, f"mIoU {E6_TAG[v]}"])

def _fill(key, v, b):
    col = [c for c in e6_fill.columns if c.startswith(b)]
    return float(e6_fill.loc[(SAM_NAME[v], key), col[0]]) if col else float("nan")

def _size(key, v, b):
    col = [c for c in e6_size.columns if c.startswith(b)]
    return float(e6_size.loc[(SAM_NAME[v], key), col[0]]) if col else float("nan")

V = SAM_VARIANTS; vS = V[-1]
best = {v: max(E6_MAIN, key=lambda k: _m(k, v)) for v in V}
best_pr = {v: max(["P1", "P2", "P3"], key=lambda k: _m(k, v)) for v in V}
thin, comp = FILL_LABELS[0].split(" ")[0], FILL_LABELS[-1].split(" ")[0]
gain_thin = {v: _fill(best[v], v, thin) - _fill("P1", v, thin) for v in V}
gain_comp = {v: _fill(best[v], v, comp) - _fill("P1", v, comp) for v in V}
n_thin = int(((e6.prompt == "P1") & (e6.method == SAM_NAME[vS]) & (e6.fill_bin == FILL_LABELS[0])).sum())
mm = e6_mm.set_index(["prompt", "model"])
find = [
    "Best prompt: " + "; ".join(f"{SAM_NAME[v]} → {E6_PROMPTS[best[v]][0]} (mIoU {_m(best[v], v):.3f} vs {_m('P1', v):.3f} for the box alone, "
                                f"{_m(best[v], v) - _m('P1', v):+.3f})" for v in V) + ".",
    "Practical prompts: a single box-centre point is far worse than the box (" + ", ".join(f"{E6_TAG[v]} {_m('P2', v):.3f}" for v in V)
    + "); box + centre point gives " + ", ".join(f"{E6_TAG[v]} {_m('P3', v):.3f} ({_m('P3', v) - _m('P1', v):+.3f})" for v in V)
    + f"; the best prompt without ground truth for {SAM_NAME[vS]} is: {E6_PROMPTS[best_pr[vS]][0]}.",
    f"Is the gain concentrated in thin objects? {'yes' if min(gain_thin.values()) > max(2 * max(gain_comp.values()), 0.01) else 'not clearly'}"
    f" — thin objects (fill < 0.3, n={n_thin}), best prompt vs box: " + ", ".join(
        f"{E6_TAG[v]} {_fill('P1', v, thin):.3f} → {_fill(best[v], v, thin):.3f} ({gain_thin[v]:+.3f})" for v in V)
    + ", compared with " + ", ".join(f"{gain_comp[v]:+.3f}" for v in V) + " on compact objects (fill ≥ 0.6). Small objects: box "
    + ", ".join(f"{E6_TAG[v]} {_size('P1', v, 'small'):.3f} → {_size(best[v], v, 'small'):.3f}" for v in V) + ".",
    "Box quality matters: mask mIoU with ±5/10/20% box jitter = " + "; ".join(
        f"{E6_TAG[v]} " + " / ".join(f"{e6_jit.loc[e6_jit['jitter %'] == l, f'mIoU box {E6_TAG[v]}'].iloc[0]:.3f}" for l in (5, 10, 20)) for v in V)
    + f" (exact box: " + ", ".join(f"{_m('P1', v):.3f}" for v in V) + "); with the jittered box's centre point added at ±20%: "
    + ", ".join(f"{e6_jit.loc[e6_jit['jitter %'] == 20, f'mIoU box+centre {E6_TAG[v]}'].iloc[0]:.3f}" for v in V) + ".",
]
if len(ov_sub):
    yb = float(e6_jit.loc[e6_jit['box prompt'] == 'YOLO box (matched)', 'mean box IoU vs GT'].iloc[0])
    find.append(f"Pipeline (n={int(ov_sub['n objects'].iloc[0])} YOLO-matched objects, mean YOLO box IoU {yb:.2f}): GT box vs YOLO box = "
                + ", ".join(f"{E6_TAG[v]} {_m('P1', v, True):.3f} vs {_m('Y1', v, True):.3f}" for v in V)
                + "; adding the YOLO box centre point → " + ", ".join(f"{E6_TAG[v]} {_m('Y3', v, True):.3f} ({_m('Y3', v, True) - _m('Y1', v, True):+.3f})" for v in V)
                + "; an oracle click on top of the GT box would reach " + ", ".join(f"{E6_TAG[v]} {_m('P5', v, True):.3f}" for v in V) + ".")
find.append("Mask selection: picking the candidate with the highest predicted IoU gives " + ", ".join(
    f"{E6_TAG[v]} {mm.loc[(E6_PROMPTS['P1'][0], SAM_NAME[v]), 'pick by predicted IoU']:.3f}" for v in V)
    + " for the box prompt, vs an oracle best-of-3 of " + ", ".join(f"{mm.loc[(E6_PROMPTS['P1'][0], SAM_NAME[v]), 'oracle best of 3']:.3f}" for v in V)
    + " (single point: " + ", ".join(f"{mm.loc[(E6_PROMPTS['P4'][0], SAM_NAME[v]), 'pick by predicted IoU']:.3f} vs "
                                    f"{mm.loc[(E6_PROMPTS['P4'][0], SAM_NAME[v]), 'oracle best of 3']:.3f}" for v in V) + ").")
print("E6 findings")
for f_ in find:
    print(" •", f_)
with open(OUT / "E6_findings.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join("- " + f_ for f_ in find) + "\n")

## 8 · E4: YOLO-seg baseline + COCO mask AP for every method

In [ ]:
def coco_eval(results, iou_type):
    if not results:
        return np.zeros(12), None
    with contextlib.redirect_stdout(io.StringIO()):
        dt = test_coco.loadRes([dict(r) for r in results])
        E = COCOeval(test_coco, dt, iou_type); E.params.imgIds = test_ids
        E.evaluate(); E.accumulate(); E.summarize()
    return E.stats, E

def per_class_ap(E):
    out = {}
    for k, cid in enumerate(E.params.catIds):
        pr = E.eval["precision"][:, :, k, 0, -1]; pr = pr[pr > -1]
        out[CLASSES[cid - 1]] = float(pr.mean()) if pr.size else float("nan")
    return out

free_gpu()
yseg_results, e4_rows = [], []
if SEG_BEST is not None:
    segm = YOLO(str(SEG_BEST))
    for i, pth in tqdm(list(zip(test_ids, paths)), desc="YOLO-seg"):      # one image per call (see E1)
        r = segm.predict(pth, conf=0.001, iou=0.7, max_det=100, imgsz=IMG_SIZE, retina_masks=True,
                         device=YOLO_DEVICE, verbose=False)[0]
        if r.masks is None or len(r.boxes) == 0:
            continue
        H, W = r.orig_shape; g = GT[i]
        md_ = r.masks.data
        if tuple(md_.shape[-2:]) != (H, W):
            md_ = F.interpolate(md_[:, None].float(), (H, W), mode="nearest")[:, 0]
        mks = (md_ > 0.5).cpu().numpy()
        sc, cl, bx = r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.xyxy.cpu().numpy()
        rl = [to_rle(mk) for mk in mks]
        for rr, s, c in zip(rl, sc, cl):
            yseg_results.append({"image_id": i, "category_id": int(c) + 1, "segmentation": rr, "score": float(s)})
        k = np.where(sc >= CONF_THRES)[0]
        pairs, _, _ = match(bx[k], sc[k], g["boxes"])
        for pk, gk in pairs:
            iou = rle_iou(rl[k[pk]], g["rles"][gk])
            e4_rows.append(dict(image_id=i, gt_idx=gk, method="YOLO-seg", cls=CLASSES[g["cls"][gk]], size=size_bin(g["area"][gk]),
                                iou=iou, dice=2 * iou / (1 + iou), class_ok=bool(cl[k[pk]] == g["cls"][gk])))
e4 = pd.DataFrame(e4_rows)

box_stats, _ = coco_eval(det_results, "bbox")
SEG_EVAL = {SAM_NAME[v]: coco_eval(seg_results[v], "segm") for v in SAM_VARIANTS}
if yseg_results:
    SEG_EVAL["YOLO-seg"] = coco_eval(yseg_results, "segm")
print(f"Detector box AP (pycocotools): AP={box_stats[0]:.3f}  AP50={box_stats[1]:.3f}  AP75={box_stats[2]:.3f}")
for k, (st, _) in SEG_EVAL.items():
    print(f"{k:16s} mask AP={st[0]:.3f}  AP50={st[1]:.3f}  AP75={st[2]:.3f}  AP_small={st[3]:.3f}  AP_medium={st[4]:.3f}  AP_large={st[5]:.3f}")

## 9 · E5: speed benchmark (end-to-end ms per image)

In [ ]:
timing_ids = test_ids[:TIMING_IMAGES]
def bench(fn):
    for i in timing_ids[:3]:
        fn(load_rgb(i))                                  # warm-up
    ts = []
    for i in timing_ids:
        img = load_rgb(i); sync(); t0 = time.perf_counter(); fn(img); sync(); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts))

def run_det(img):
    # agnostic NMS: one label per object, so the litter count is not inflated by overlapping boxes of different classes
    r = det.predict(np.ascontiguousarray(img[..., ::-1]), conf=CONF_THRES, imgsz=DET_IMGSZ, device=YOLO_DEVICE,
                    agnostic_nms=True, verbose=False)[0]
    return r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()

def make_pipeline(v):
    def fn(img):
        boxes, cls, sc = run_det(img)
        H, W = img.shape[:2]
        masks = esam_decode(ESAM[v], esam_embed(ESAM[v], img), boxes, H, W) if len(boxes) else np.zeros((0, H, W), bool)
        return boxes, cls, sc, masks
    return fn

SPEED = {"YOLO detector only": bench(run_det)}
for v in SAM_VARIANTS:
    SPEED[f"YOLO + {SAM_NAME[v]}"] = bench(make_pipeline(v))
if SEG_BEST is not None:
    SPEED["YOLO-seg"] = bench(lambda img: segm.predict(np.ascontiguousarray(img[..., ::-1]), conf=CONF_THRES, imgsz=IMG_SIZE,
                                                       retina_masks=True, device=YOLO_DEVICE, verbose=False))
for k, v in SPEED.items():
    print(f"{k:24s} {v:7.1f} ms / image")
print({SAM_NAME[v]: f"encoder {np.mean(enc_ms[v]):.1f} ms" for v in SAM_VARIANTS})

## 10 · Results tables

In [ ]:
rows = []
for v in SAM_VARIANTS:
    n = SAM_NAME[v]; d2 = e2[e2.method == n]
    rows.append([f"GT box → {n}", "oracle box", np.nan, np.nan, np.nan, d2.iou.mean(), d2.dice.mean(), (d2.iou >= 0.5).mean(),
                 np.nan, PARAMS[n]])
for v in SAM_VARIANTS:
    n = SAM_NAME[v]; st = SEG_EVAL[n][0]; d3 = e3[e3.method == n] if len(e3) else e3
    rows.append([f"YOLO box → {n}", "predicted box", st[0], st[1], st[2],
                 d3.iou.mean() if len(d3) else np.nan, d3.dice.mean() if len(d3) else np.nan,
                 (d3.iou >= 0.5).mean() if len(d3) else np.nan, SPEED[f"YOLO + {n}"], PARAMS[n] + PARAMS["YOLO detector"]])
if "YOLO-seg" in SEG_EVAL:
    st = SEG_EVAL["YOLO-seg"][0]
    rows.append(["YOLO-seg (one-stage)", "—", st[0], st[1], st[2], e4.iou.mean() if len(e4) else np.nan,
                 e4.dice.mean() if len(e4) else np.nan, (e4.iou >= 0.5).mean() if len(e4) else np.nan, SPEED["YOLO-seg"],
                 sum(p.numel() for p in segm.model.parameters()) / 1e6])
main_table = pd.DataFrame(rows, columns=["Method", "Prompt", "Mask AP", "Mask AP50", "Mask AP75", "mIoU", "mDice",
                                         "IoU≥0.5 rate", "ms / image", "Params (M)"]).set_index("Method")
main_table = main_table.round(3); main_table.to_csv(OUT / "TABLE_main_segmentation_results.csv")
print("Main comparison (test split). mIoU/mDice: E2 = all GT objects; E3/E4 = detections matched at conf ≥", CONF_THRES)
display(main_table)

e2_cls = e2.pivot_table(index="cls", columns="method", values="iou", aggfunc="mean").reindex(CLASSES)
e2_size = e2.pivot_table(index="size", columns="method", values="iou", aggfunc="mean").reindex(["small", "medium", "large"])
e2_size["objects"] = e2[e2.method == SAM_NAME[SAM_VARIANTS[0]]].groupby("size").size().reindex(e2_size.index)
ap_cls = pd.DataFrame({k: per_class_ap(E) for k, (st, E) in SEG_EVAL.items() if E is not None}).reindex(CLASSES)
for name, t in [("TABLE_E2_iou_by_class", e2_cls), ("TABLE_E2_iou_by_size", e2_size), ("TABLE_mask_AP_by_class", ap_cls)]:
    t.round(3).to_csv(OUT / f"{name}.csv"); print(name); display(t.round(3))

## 11 · Result charts

In [ ]:
def grouped_bars(ax, table, methods, ylabel, title, ylim=(0, 1)):
    x = np.arange(len(table.index)); w = 0.8 / len(methods)
    for k, mth in enumerate(methods):
        vals = table[mth].values
        bars = ax.bar(x + (k - (len(methods) - 1) / 2) * w, vals, w * 0.92, label=mth, color=METHOD_COLORS[mth])
        for b_, v_ in zip(bars, vals):
            if not np.isnan(v_):
                ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.01, f"{v_:.2f}", ha="center", va="bottom", fontsize=7.5, color=INK2)
    ax.set_xticks(x, table.index, rotation=0); ax.set_ylim(*ylim); ax.set_ylabel(ylabel); ax.set_title(title, loc="left")
    if ylim[1] > 1:                                   # headroom above 1.0 is only for the legend
        ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.legend(fontsize=8, ncol=len(methods), loc="upper left")

sam_methods = [SAM_NAME[v] for v in SAM_VARIANTS]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
grouped_bars(ax[0], e2_cls, sam_methods, "mean IoU", "E2 · GT box prompts: mask IoU by class", (0, 1.22))
grouped_bars(ax[1], e2_size.drop(columns="objects"), sam_methods, "mean IoU", "E2 · GT box prompts: mask IoU by object size", (0, 1.22))
ax[1].set_xticks(range(3), [f"{s}\n(n={int(n)})" for s, n in zip(e2_size.index, e2_size["objects"].fillna(0))])
plt.tight_layout(); savefig("fig_E2_iou_by_class_and_size.png")

fig, ax = plt.subplots(figsize=(10, 4))
grouped_bars(ax, ap_cls, list(ap_cls.columns), "mask AP (0.50:0.95)", "Mask AP by class: YOLO→EfficientSAM vs YOLO-seg", (0, min(1.0, max(0.1, np.nanmax(ap_cls.values) * 1.4))))
plt.tight_layout(); savefig("fig_mask_AP_by_class.png")

# prompt quality: GT box vs YOLO box (same objects only)
if len(e3):
    both = e3.merge(e2, on=["image_id", "gt_idx", "method"], suffixes=("_yolo", "_gt"))
    comp = both.groupby("method")[["iou_gt", "iou_yolo"]].mean().reindex(sam_methods)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    x = np.arange(len(comp)); w = 0.36
    for k, (col, lab, c) in enumerate([("iou_gt", "GT box prompt", "#2a78d6"), ("iou_yolo", "YOLO box prompt", "#eb6834")]):
        bars = ax.bar(x + (k - 0.5) * w, comp[col], w * 0.92, label=lab, color=c)
        for b_, v_ in zip(bars, comp[col]):
            ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.01, f"{v_:.3f}", ha="center", fontsize=8, color=INK2)
    ax.set_xticks(x, comp.index); ax.set_ylim(0, 1.22); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel("mean IoU (same objects)")
    ax.set_title(f"Effect of the box prompt (n={len(both) // max(len(sam_methods), 1)} matched objects)", loc="left"); ax.legend(fontsize=8, ncol=2, loc="upper left")
    plt.tight_layout(); savefig("fig_E3_gt_vs_yolo_prompt.png")
    comp.round(3).to_csv(OUT / "TABLE_E3_gt_vs_yolo_prompt.csv")

# speed vs accuracy
pts = [(f"YOLO + {SAM_NAME[v]}", SPEED[f"YOLO + {SAM_NAME[v]}"], SEG_EVAL[SAM_NAME[v]][0][0], METHOD_COLORS[SAM_NAME[v]]) for v in SAM_VARIANTS]
if "YOLO-seg" in SEG_EVAL:
    pts.append(("YOLO-seg", SPEED["YOLO-seg"], SEG_EVAL["YOLO-seg"][0][0], METHOD_COLORS["YOLO-seg"]))
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, ms, ap, c in pts:
    ax.scatter(ms, ap, s=90, color=c, edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(f"{name}\n{ap:.3f} AP · {ms:.0f} ms", (ms, ap), textcoords="offset points", xytext=(8, 6), fontsize=8, color=INK2)
ax.grid(axis="both"); ax.set_xlabel("ms per image (end-to-end, " + ("GPU" if DEVICE == "cuda" else "CPU") + ")"); ax.set_ylabel("mask AP")
ax.set_xlim(0, max(p[1] for p in pts) * 1.45); ax.set_ylim(0, max(0.05, max(p[2] for p in pts) * 1.35))
ax.set_title("Speed vs accuracy (test split)", loc="left")
plt.tight_layout(); savefig("fig_speed_vs_accuracy.png")

## 12 · Qualitative results: correct and incorrect predictions

In [ ]:
def crop_box(b, H, W, pad=0.3, mn=40):
    x0, y0, x1, y1 = b; w, h = x1 - x0, y1 - y0; p = pad * max(w, h) + 8
    return int(max(0, x0 - p)), int(max(0, y0 - p)), int(min(W, x1 + p)), int(min(H, y1 + p))

# ---- EfficientSAM best / worst masks (GT box prompts) ----
ref = SAM_NAME[SAM_VARIANTS[-1]]
cand = e2[(e2.method == ref) & (e2.area >= 40 ** 2)].sort_values("iou")
picks = list(cand.itertuples())[::-1] if len(cand) <= 8 else list(cand.tail(4).itertuples())[::-1] + list(cand.head(4).itertuples())
if picks:
    fig, axes = plt.subplots(1 + len(SAM_VARIANTS), len(picks), figsize=(2.1 * len(picks), 2.2 * (1 + len(SAM_VARIANTS))))
    axes = np.atleast_2d(axes)
    for j, rw in enumerate(picks):
        img = load_rgb(rw.image_id); H, W = img.shape[:2]; g = GT[rw.image_id]
        x0, y0, x1, y1 = crop_box(g["boxes"][rw.gt_idx], H, W)
        gt_mask = mask_utils.decode(g["rles"][rw.gt_idx]).astype(bool)
        panels = [("GT", gt_mask, "#ffffff")] + [(SAM_NAME[v], mask_utils.decode(rles_e2[(rw.image_id, rw.gt_idx, v)]).astype(bool), METHOD_COLORS[SAM_NAME[v]]) for v in SAM_VARIANTS]
        for r_, (nm, mk, col) in enumerate(panels):
            a_ = axes[r_, j]; a_.axis("off")
            vis = overlay(img, mk, col if nm != "GT" else "#1baf7a")
            bx = g["boxes"][rw.gt_idx].astype(int); cv2.rectangle(vis, tuple(bx[:2]), tuple(bx[2:]), (255, 255, 255), 2)
            a_.imshow(vis[y0:y1, x0:x1])
            if nm == "GT":
                tag = "✓ good" if rw.iou >= 0.75 else ("✗ bad" if rw.iou < 0.5 else "~ fair")
                a_.set_title(f"{tag} · {rw.cls}", fontsize=8, color=INK)
            else:
                iou = e2[(e2["image_id"] == rw.image_id) & (e2["gt_idx"] == rw.gt_idx) & (e2["method"] == nm)]["iou"].iloc[0]
                a_.set_title(f"{nm.replace('EfficientSAM', 'ESAM')}  IoU {iou:.2f}", fontsize=8, color=INK2)
    plt.suptitle("EfficientSAM masks from GT box prompts: 4 best vs 4 worst objects (≥40×40 px)", fontweight="bold", y=1.01)
    plt.tight_layout(); savefig("fig_E2_good_vs_bad_masks.png")

# ---- detector: correct vs incorrect images ----
def draw_det(img, i):
    vis = img.copy(); pim, g = per_image[i], GT[i]
    fs, th = 0.55 * max(1.0, img.shape[1] / 700), max(2, round(img.shape[1] / 450))
    matched_p = {p: gk for p, gk in pim["pairs"]}
    for p in range(len(pim["pb"])):
        b = pim["pb"][p].astype(int); c = pim["pc"][p]
        if p in matched_p and c == g["cls"][matched_p[p]]:
            col, txt = (12, 163, 12), CLASSES[c]                        # correct
        elif p in matched_p:
            col, txt = (208, 59, 59), f"{CLASSES[c]} (true: {CLASSES[g['cls'][matched_p[p]]]})"   # wrong class
        else:
            col, txt = (208, 59, 59), f"FP {CLASSES[c]}"                # false positive
        cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), col, th + 1)
        cv2.putText(vis, txt, (b[0], max(int(25 * fs), b[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, col, th, cv2.LINE_AA)
    for gk in pim["umg"]:
        b = g["boxes"][gk].astype(int)
        cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), (250, 178, 25), th + 1)
        cv2.putText(vis, f"missed {CLASSES[g['cls'][gk]]}", (b[0], max(int(25 * fs), b[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, (250, 178, 25), th, cv2.LINE_AA)
    return vis

scored = []
for i in test_ids:
    pim = per_image[i]; errs = len(pim["ump"]) + len(pim["umg"]) + (len(pim["pairs"]) - pim["n_ok"])
    scored.append((errs, pim["n_ok"], i))
good = [i for e, n, i in sorted(scored, key=lambda t: (t[0], -t[1])) if e == 0 and n > 0][:3]
bad = [i for e, n, i in sorted(scored, key=lambda t: -t[0]) if e > 0][:3]
sel = [(i, "✓ all correct") for i in good] + [(i, "✗ contains errors") for i in bad]
if sel:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5)); axes = axes.ravel()
    for a_ in axes: a_.axis("off")
    for a_, (i, t) in zip(axes, sel):
        a_.imshow(draw_det(load_rgb(i), i)); a_.set_title(t, loc="left", fontsize=11)
    plt.suptitle(f"Detector on test images (conf ≥ {CONF_THRES}) · green = correct · red = wrong class / false positive · yellow = missed",
                 fontweight="bold", y=1.0)
    plt.tight_layout(); savefig("fig_detection_correct_vs_wrong.png")

In [ ]:
# ---- full pipeline output: what the prototype shows ----
def draw_instances(img, boxes, cls, scores, masks):
    vis = img.copy()
    fs, th = 0.5 * max(1.0, img.shape[1] / 700), max(1, round(img.shape[1] / 600))
    for b, c, s, mk in sorted(zip(boxes, cls, scores, masks), key=lambda t: -t[3].sum()):   # big masks first
        vis = overlay(vis, mk, CLASS_COLORS[c], alpha=0.5)
        b = np.asarray(b).astype(int); col = hex2rgb(CLASS_COLORS[c])
        cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), col, th + 1)
        lab = CLASSES[c] if s is None else f"{CLASSES[c]} {s:.2f}"
        (tw, tht), _ = cv2.getTextSize(lab, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
        cv2.rectangle(vis, (b[0], max(0, b[1] - tht - 10)), (b[0] + tw + 8, max(tht + 10, b[1])), col, -1)
        cv2.putText(vis, lab, (b[0] + 4, max(tht + 4, b[1] - 5)), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), th, cv2.LINE_AA)
    return vis

SHORT = {"Plastic": "Plastic", "Paper & Cardboard": "Paper", "Metal": "Metal", "Glass": "Glass", "Other Waste": "Other"}
def count_text(cls):
    c = Counter(CLASSES[k] for k in cls)
    items = [f"{SHORT[k]} {c[k]}" for k in CLASSES if c[k]] + [f"total {len(cls)}"]
    return "\n".join(" · ".join(items[k:k + 2]) for k in range(0, len(items), 2))   # 2 per line, fits narrow photos

pipe = make_pipeline(SAM_VARIANTS[-1])
show = [i for _, _, i in sorted(scored, key=lambda t: (t[0], -t[1]))[:4]]
if show:
    fig, axes = plt.subplots(2, len(show), figsize=(4.4 * len(show), 7.5)); axes = np.atleast_2d(axes).reshape(2, -1)
    for j, i in enumerate(show):
        img = load_rgb(i); g = GT[i]; H, W = img.shape[:2]
        gt_masks = [mask_utils.decode(r).astype(bool) for r in g["rles"]]
        axes[0, j].imshow(draw_instances(img, g["boxes"], g["cls"], [None] * len(g["cls"]), gt_masks)); axes[0, j].axis("off")
        axes[0, j].set_title("Ground truth\n" + count_text(g["cls"]), fontsize=9, loc="left")
        boxes, cls, sc, masks = pipe(img)
        axes[1, j].imshow(draw_instances(img, boxes, cls, sc, masks)); axes[1, j].axis("off")
        axes[1, j].set_title(f"YOLO + {SAM_NAME[SAM_VARIANTS[-1]]}\n" + count_text(cls), fontsize=9, loc="left")
    handles = [matplotlib.patches.Patch(color=c, label=n) for n, c in zip(CLASSES, CLASS_COLORS)]
    fig.legend(handles=handles, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.03), fontsize=9)
    plt.tight_layout(); savefig("fig_pipeline_examples.png")

## 12.5 · Benchmark: are we higher or lower than published work?
No published work uses exactly our 5 material classes and 70/15/15 split, so these are the **closest** comparisons, each with its caveat
(full citations and the table each number comes from: `final/benchmark/benchmark_notes.md`). Published numbers are fixed; our rows are filled in from this run.

In [ ]:
BENCH = [  # (group, source, setting, model, value, caveat)
    ("Box mAP50 (multi-class TACO)", "Majchrowska et al. 2022, Waste Management", "TACO+ · 7 material classes", "EfficientDet-D2", 0.162, "more images (≈4.5k), 7 classes"),
    ("Box mAP50 (multi-class TACO)", "Saakyan & Solntsev 2026, arXiv 2602.23043", "TACO · 59 classes · 640 px", "YOLO26-S", 0.124, "59 fine classes (harder)"),
    ("Box mAP50 (multi-class TACO)", "Saakyan & Solntsev 2026, arXiv 2602.23043", "TACO · 59 classes · 640 px", "D-FINE-S", 0.244, "59 fine classes (harder)"),
    ("Box mAP50 (1-class 'litter')", "Córdova et al. 2022, Sensors", "TACO · 1 class", "YOLOv5s @640", 0.547, "no classification needed (easier)"),
    ("Mask AP (TACO)", "Proença & Simões 2020 (TACO paper)", "TACO-10 · 4-fold CV", "Mask R-CNN R50", 0.176, "10 classes, ±0.016 over folds"),
    ("Mask AP (TACO)", "Saakyan & Solntsev 2026, arXiv 2602.23043", "TACO · 59 classes · 640 px", "YOLO26-seg S", 0.111, "59 fine classes"),
    ("Mask AP (TACO)", "Saakyan & Solntsev 2026, arXiv 2602.23043", "TACO · 59 classes · 640 px", "D-FINE-seg S", 0.177, "59 fine classes"),
    ("Box-prompt mIoU (GT box)", "Xiong et al. 2024, EfficientSAM (CVPR)", "COCO", "EfficientSAM-Ti", 0.757, "reference paper; COCO not TACO"),
    ("Box-prompt mIoU (GT box)", "Xiong et al. 2024, EfficientSAM (CVPR)", "COCO", "EfficientSAM-S", 0.769, "reference paper; COCO not TACO"),
    ("Box-prompt mIoU (GT box)", "Xiong et al. 2024, EfficientSAM (CVPR)", "COCO", "SAM ViT-H (636M)", 0.784, "reference paper; 25–60× bigger"),
]
ours = []
for tag, t in [("v1 (progress)", det_table_v1)] + ([("v2 (final)", det_table_v2)] if DET_V2_BEST is not None else []):
    ours.append(("Box mAP50 (multi-class TACO)", "OURS", "TACO · 5 material classes", f"YOLO11s {tag}", float(t.loc["all", "mAP50"]), ""))
for k, (st, _) in SEG_EVAL.items():
    name = f"YOLO → {k}" if k != "YOLO-seg" else "YOLO11s-seg (one-stage)"
    ours.append(("Mask AP (TACO)", "OURS", "TACO · 5 material classes", name, float(st[0]), "detector " + ("v2" if DET_V2_BEST is not None else "v1")))
for v in SAM_VARIANTS:
    ours.append(("Box-prompt mIoU (GT box)", "OURS", "TACO test (710 objects)", SAM_NAME[v], float(e2[e2.method == SAM_NAME[v]].iou.mean()), "zero-shot"))
bench = pd.DataFrame(BENCH + ours, columns=["metric", "source", "setting", "model", "value", "caveat"])
bench["value"] = bench["value"].round(3)
bench.to_csv(OUT / "TABLE_benchmark_comparison.csv", index=False)
display(bench)

groups = list(dict.fromkeys(bench.metric))
fig, axes = plt.subplots(1, len(groups), figsize=(4.6 * len(groups), 4.6))
for ax, g in zip(axes, groups):
    d = bench[bench.metric == g].sort_values("value")
    cols = ["#eb6834" if s == "OURS" else "#c3c2b7" for s in d.source]
    ax.barh(range(len(d)), d.value, color=cols, height=0.65)
    ax.set_yticks(range(len(d)), [m if s == "OURS" else f"{m}\n{s.split(',')[0]}" for m, s in zip(d.model, d.source)], fontsize=7.5)
    for y, v in enumerate(d.value):
        ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=8, color=INK2)
    ax.set_xlim(0, max(0.1, d.value.max() * 1.3)); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.set_title(g, loc="left", fontsize=10)
fig.legend(handles=[matplotlib.patches.Patch(color="#eb6834", label="ours"), matplotlib.patches.Patch(color="#c3c2b7", label="published")],
           loc="upper right", ncol=2, fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.94)); savefig("fig_benchmark_comparison.png")

our_map = bench[(bench.source == "OURS") & (bench.metric == "Box mAP50 (multi-class TACO)")].value.iloc[-1]   # final detector (v2 if trained)
our_ap = bench[(bench.source == "OURS") & (bench.metric == "Mask AP (TACO)") & bench.model.str.startswith("YOLO →")].value.max()  # best YOLO→EfficientSAM
print(f"• Box mAP50: ours {our_map:.3f} vs closest material-class benchmark 0.162 (7 classes) → {'higher' if our_map > 0.162 else 'lower'}; "
      f"vs 1-class 'litter' detectors 0.547 → lower (they do not classify).")
print(f"• Mask AP: ours {our_ap:.3f} vs TACO paper Mask R-CNN 0.176 ± 0.016 (TACO-10) → {'higher' if our_ap > 0.176 + 0.016 else 'about the same'}.")
print(f"• EfficientSAM on TACO with GT boxes ({', '.join(f'{SAM_NAME[v]} {e2[e2.method == SAM_NAME[v]].iou.mean():.3f}' for v in SAM_VARIANTS)}) "
      f"matches the reference paper on COCO (0.757 / 0.769): zero-shot transfer to litter works; the detector is the bottleneck.")

## 13 · Summary for the slides + download everything

In [ ]:
best_sam = max((SAM_NAME[v] for v in SAM_VARIANTS), key=lambda n: SEG_EVAL[n][0][0])
lines = ["# Results summary (auto-generated)", "",
         f"- Test split: {len(test_ids)} images, {sum(len(GT[i]['cls']) for i in test_ids)} objects; detector `{DET_V2_MODEL if DET_V2_BEST is not None else DET_MODEL}` at {DET_IMGSZ}px ({"v2: RFS + 960 px" if DET_V2_BEST is not None else "v1"})",
         f"- Detector (test): P={det_table.loc['all','Precision']:.3f}  R={det_table.loc['all','Recall']:.3f}  F1={det_table.loc['all','F1']:.3f}  "
         f"mAP50={det_table.loc['all','mAP50']:.3f}  mAP50-95={det_table.loc['all','mAP50-95']:.3f}",
         f"- Classification accuracy of localised objects (conf ≥ {CONF_THRES}): {cls_acc:.3f}",
         f"- E2 GT-box mIoU: " + ", ".join(f"{SAM_NAME[v]} {e2[e2.method == SAM_NAME[v]].iou.mean():.3f}" for v in SAM_VARIANTS),
         f"- E3 pipeline mask AP: " + ", ".join(f"{k} {st[0]:.3f}" for k, (st, _) in SEG_EVAL.items()),
         f"- Best EfficientSAM variant on mask AP: {best_sam}",
         f"- Speed: " + ", ".join(f"{k} {v:.0f} ms" for k, v in SPEED.items()), ""]
def md_table(df):
    try:
        return df.to_markdown()
    except Exception:
        return "```\n" + df.to_string() + "\n```"
for title, df in [("E1 detection (Ultralytics, test)", det_table), (f"E1 detection at conf ≥ {CONF_THRES}", thr_table),
                  ("Main segmentation comparison", main_table), ("E2 mean IoU by class (GT boxes)", e2_cls.round(3)),
                  ("E2 mean IoU by object size (GT boxes)", e2_size.round(3)), ("Mask AP by class", ap_cls.round(3))]:
    lines += [f"## {title}", "", md_table(df), ""]
# final-only tables (E1 v1 vs v2, E6 prompts, E7 imbalance metrics, benchmark)
for title, f in [("E1 detector v1 vs v2 (Ultralytics, test)", "TABLE_E1_v1_vs_v2.csv"),
                 ("E7 imbalance-aware detector comparison", "TABLE_E7_detector_comparison.csv"),
                 ("E7 object-level ROC-AUC / PR-AUC", "TABLE_E7_auc.csv"),
                 ("E6 prompt study (overall)", "TABLE_E6_prompt_overall.csv"),
                 ("E6 prompt study by fill ratio (thin objects)", "TABLE_E6_prompt_by_fill.csv"),
                 ("Benchmark comparison", "TABLE_benchmark_comparison.csv")]:
    if (OUT / f).exists():
        t = pd.read_csv(OUT / f, header=[0, 1], index_col=0) if f == "TABLE_E1_v1_vs_v2.csv" else pd.read_csv(OUT / f)
        if isinstance(t.columns, pd.MultiIndex):
            t.columns = [f"{m} {d}" for m, d in t.columns]
        lines += [f"## {title}", "", md_table(t), ""]
(OUT / "results_summary.md").write_text("\n".join(lines))
print("\n".join(lines[:10]))

# weights for the prototype
(OUT / "weights").mkdir(exist_ok=True)
shutil.copy(DET_BEST, OUT / "weights/yolo_det_best.pt")          # the pipeline detector (v2)
shutil.copy(DET_V1_BEST, OUT / "weights/yolo_det_v1.pt")
if SEG_BEST is not None:
    shutil.copy(SEG_BEST, OUT / "weights/yolo_seg_best.pt")
zip_path = shutil.make_archive(str(WORK / "outputs"), "zip", OUT)
print("\n📦", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.0f} MB)")
print("figures:", sorted(p.name for p in FIG.glob("*.png")))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 14 · Prototype: web demo (Gradio)
Upload a photo to get boxes, classes, EfficientSAM masks and a litter count. Open the public `gradio.live` link it prints, or use the inline UI.

In [ ]:
def predict_image(img, conf=CONF_THRES, variant=SAM_NAME[SAM_VARIANTS[-1]]):
    if img is None:
        return None, "Upload an image"
    img = np.asarray(img)[..., :3].astype(np.uint8)
    s = min(1.0, MAX_SIDE / max(img.shape[:2]))
    if s < 1:
        img = cv2.resize(img, (round(img.shape[1] * s), round(img.shape[0] * s)), interpolation=cv2.INTER_AREA)
    v = {n: k for k, n in SAM_NAME.items()}[variant]
    r = det.predict(np.ascontiguousarray(img[..., ::-1]), conf=conf, imgsz=DET_IMGSZ, device=YOLO_DEVICE,
                    agnostic_nms=True, iou=0.5, verbose=False)[0]
    boxes, cls, sc = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
    keep = [i for i in np.argsort(-sc)]                # demo only: drop a box that lies ≥ 80% inside a higher-scoring box
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]); kept = []
    for i in keep:
        iw = np.clip(np.minimum(boxes[i, 2], boxes[kept, 2]) - np.maximum(boxes[i, 0], boxes[kept, 0]), 0, None)
        ih = np.clip(np.minimum(boxes[i, 3], boxes[kept, 3]) - np.maximum(boxes[i, 1], boxes[kept, 1]), 0, None)
        if not len(kept) or not (iw * ih >= 0.8 * np.minimum(area[i], area[kept])).any():
            kept.append(i)
    kept = sorted(kept); boxes, cls, sc = boxes[kept], cls[kept], sc[kept]
    H, W = img.shape[:2]
    masks = esam_decode(ESAM[v], esam_embed(ESAM[v], img), boxes, H, W) if len(boxes) else np.zeros((0, H, W), bool)
    c = Counter(CLASSES[k] for k in cls)
    return draw_instances(img, boxes, cls, sc, masks), "\n".join(f"{k}: {c[k]}" for k in CLASSES) + f"\n\nTotal: {len(cls)}"

# quick check on one test image
_vis, _txt = predict_image(load_rgb(test_ids[0]))
plt.figure(figsize=(7, 5)); plt.imshow(_vis); plt.axis("off"); plt.title(_txt.replace("\n", "  "), fontsize=8, loc="left"); plt.show()

if LAUNCH_DEMO:
    try:
        import gradio as gr
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
        import gradio as gr
    demo = gr.Interface(
        fn=predict_image,
        inputs=[gr.Image(type="numpy", label="Litter photo"),
                gr.Slider(0.05, 0.9, value=CONF_THRES, step=0.05, label="Detection confidence"),
                gr.Radio([SAM_NAME[v] for v in SAM_VARIANTS], value=SAM_NAME[SAM_VARIANTS[-1]], label="Segmenter")],
        outputs=[gr.Image(label="Boxes + EfficientSAM masks"), gr.Textbox(label="Litter count")],
        examples=[[str(FAST_IMG / images_all[i]["file_name"]), CONF_THRES, SAM_NAME[SAM_VARIANTS[-1]]] for i in test_ids[:4]],
        title="Automatic Litter Detection & Segmentation · YOLO + EfficientSAM (final)",
        description="Team อุดมไปด้วยธาตุเหล็ก · CP461. Detects 5 litter classes, segments each object with EfficientSAM and counts them.")
    demo.launch(share=True, debug=False)